# Extração de Features — Tese

Três variantes de extração de features a partir do dataset consolidado MIMIC (ECG + PPG + ABP). Cada variante tem a sua secção de **extração** e de **visualização** dos sinais/deteções.

Ficheiros guardados em `/media/filipe/Dados/Features/` (o dataset bruto continua em `/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/` — só as features mudaram de sítio).

| Variante | Ficheiro de features | Notas |
|---|---|---|
| 1. Baseline | `features_*_baseline.csv` | Teste 1 (deteção R/T/PPG corrigida — ver 1.1/1.2) |
| 2. Com SQI | `features_*_com_sqi.csv` | SQI de skewness + retenção adaptativa por paciente |
| 3. Com NeuroKit | `features_*_com_nk.csv` | features via NeuroKit2 |


## 1. Baseline (Teste final 1)

### 1.1 Extração

In [ ]:
import os
import pandas as pd
import wfdb
import numpy as np
import re
from scipy.signal import find_peaks, butter, filtfilt
from scipy.integrate import simpson
from scipy.stats import skew
from joblib import Parallel, delayed
import multiprocessing

# --- CONFIGURAÇÃO ---
PATH_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"   # onde está o dataset bruto (leitura)
INPUT_CSVS = [f"{PATH_DATASET}/dataset_para_extração_features.csv"]
PATH_FEATURES_OUT = "/media/filipe/Dados/Features"               # onde gravar as features (escrita)
WINDOW_SIZE_SEC = 15
STRIDE_SEC = 5  # avanço de 5 s: as janelas de 15 s sobrepõem-se

# Nº de processos paralelos. O pico de RAM ≈ N_JOBS × (registo maior a ser processado). Alguns
# registos têm >1.5 GB; usar todos os cores (16) fazia colidir vários registos grandes e esgotar a
# RAM. 8 é um compromisso seguro (≈ metade dos cores). Pode subir-se com RAM folgada
# e descer-se se voltar a estourar. `None` = todos os cores (comportamento antigo, arriscado com este dataset).
N_JOBS = 16

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def butter_highpass_filter(data, cutoff, fs, order=4):
    """Remove flutuações respiratórias da linha de base do ECG (Baseline Wander)"""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, data)

def remover_picos_isolados(sinal, limiar=0.15):
    """Repara glitches de UMA amostra (ex: dropout do ADC): marca a amostra i só se AMBOS os
    vizinhos imediatos divergem muito dela mas concordam ENTRE si — um QRS genuíno nunca tem essa
    assinatura (a subida ocupa várias amostras, vizinhos não concordam entre si). Substitui pela
    média dos vizinhos. Um filtro de mediana/MAD com janela maior corta os próprios picos R."""
    s = np.asarray(sinal, dtype=float)
    if len(s) < 3:
        return s
    d_ant = np.abs(s[1:-1] - s[:-2])
    d_prox = np.abs(s[1:-1] - s[2:])
    d_vizinhos = np.abs(s[:-2] - s[2:])
    glitch = (d_ant > limiar) & (d_prox > limiar) & (d_vizinhos < limiar * 0.5)
    limpo = s.copy()
    idx = np.where(glitch)[0] + 1
    limpo[idx] = (s[idx - 1] + s[idx + 1]) / 2
    return limpo

# Canal de ECG: EXCLUSIVAMENTE a derivação II (nome exato, não substring — 'V'/'I' soltos
# apanhavam CVP/ICP/IC2 por conterem essas letras). Nada de fallback para outras derivações: o
# setup real do projeto (AD8232, 3 elétrodos) mede Lead II, e treinar com a morfologia de QRS/T
# de outra derivação criaria uma discrepância treino↔uso real. Neste dataset todos os 9646
# registos têm II, por isso esta restrição não perde nenhum paciente.
def selecionar_ecg(sig_names):
    return sig_names.index('II') if 'II' in sig_names else None

def extrair_targets_abp_robusto(win_abp, fs):
    sbp_idxs, _ = find_peaks(win_abp, distance=int(fs*0.4), height=np.percentile(win_abp, 50))
    dbp_idxs, _ = find_peaks(-win_abp, distance=int(fs*0.4))

    if len(sbp_idxs) < 3 or len(dbp_idxs) < 3:
        return None, None, None, None

    sbp_vals = win_abp[sbp_idxs]
    sbp_mean = np.mean(sbp_vals)
    sbp_std = np.std(sbp_vals)
    dbp_mean = np.mean(win_abp[dbp_idxs])
    pulse_pressure = sbp_mean - dbp_mean

    return sbp_mean, dbp_mean, sbp_std, pulse_pressure


# ========== BEAT VAZIO ==========
def beat_vazio(r_now, q_idx, s_idx, t_idx, qn_idx, qrs_ok, qrs_duration, r_amplitude,
               qrs_amplitude, qrs_area, qt_interval, tq_interval, qtc_interval, sdi_interaction, hr):
    """Batimento sem pico PPG correspondente (ppg_pk é None). As métricas do lado ECG (Q/R/S/T,
    duração/amplitude do QRS, HR) já foram calculadas antes deste ponto e são independentes de
    haver pulso PPG a seguir — preservá-las evita marcar um QRS válido como 'excluído' só porque
    o PPG falhou (ex.: pulso fora da janela, ruído momentâneo no PPG)."""
    return {
        'r_idx': int(r_now), 'r_next_idx': None, 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
        'ppg_pk_idx': None, 'ppg_vly_idx': None, 'pat': None, 'hr': hr, 'pat_kept': False, 'qrs_ok': qrs_ok,
        'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude, 'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area,
        'qt_interval': qt_interval, 'tq_interval': tq_interval, 'qtc_interval': qtc_interval, 'sdi_interaction': sdi_interaction,
        'crest_time': None, 'decay_time': None, 'alpha': None, 'alpha_b': None, 'pulse_area': None, 'w50': None,
        'ppg_valley_intensity': None, 'ppg_ac_amplitude': None
    }


# ========== PIPELINE TESTE 1 (SEM NEUROKIT) — partilhado entre baseline/com_sqi/sem_sqi ==========
def processar_janela_teste1(win_ecg, win_ppg, fs):
    """Processa UMA janela com a lógica do Teste 1 (scipy, sem NeuroKit).
        win_ecg : ECG já filtrado (highpass 0.5 Hz, remove baseline wander)
        win_ppg : PPG já filtrado (lowpass 8 Hz)
    Devolve (beats, features, win_ecg, win_ppg). Os índices dos batimentos são relativos à janela.
    'r_next_idx' guarda o fim do pulso PPG (pe)."""
    win_ppg_pos = win_ppg - np.min(win_ppg) + 0.01   # positividade p/ alpha (não move os índices)

    # Deteção de picos R — abordagem HÍBRIDA (amplitude do ECG + envelope de Pan-Tompkins):
    #  • Candidatos por AMPLITUDE no próprio ECG (picos positivos ≥15% do máximo). Isto apanha
    #    de forma fiável todos os R genuínos, mesmo os de QRS menos afiado — um detetor puramente
    #    baseado no envelope (threshold global 0.3*max) perdia R verdadeiros quando o máximo do
    #    envelope estava inflado por uma onda S muito abrupta noutro ponto da janela.
    #  • O envelope (derivada²+integração ~80ms, realça a subida/descida rápida do QRS e suprime a
    #    onda T, mais lenta) serve para DESEMPATAR: quando dois candidatos ficam a <0.4s, fica o de
    #    maior envelope local (o R, sempre mais afiado) — mesmo que a onda T seja MAIS ALTA que o R
    #    (R atenuado). Assim resolve-se o problema clássico de marcar a T como R sem usar amplitude.
    deriv = np.diff(win_ecg, prepend=win_ecg[0]) * fs
    win_integ = max(1, int(0.08 * fs))
    envelope = np.convolve(deriv ** 2, np.ones(win_integ) / win_integ, mode='same')

    def _env_local(p):
        lo, hi = max(0, p - int(0.05 * fs)), min(len(envelope), p + int(0.05 * fs))
        return np.max(envelope[lo:hi])

    candidatos, _ = find_peaks(win_ecg, height=0.15 * np.max(win_ecg))
    r_peaks = []
    for p in candidatos:
        if r_peaks and (p - r_peaks[-1]) < int(fs * 0.4):
            env_p, env_last = _env_local(p), _env_local(r_peaks[-1])
            # Desempate: só troca por ENVELOPE quando a diferença é clara (>15%) — é o caso T-vs-R
            # (uma T mesmo alta tem sempre envelope MUITO menor, tipicamente >10x, porque é lenta).
            # Quando os envelopes ficam próximos, os dois candidatos pertencem à MESMA subida aguda
            # do QRS (ex.: um "ombro"/ruído no meio da subida antes do topo verdadeiro) — as janelas
            # de +-50ms usadas por _env_local sobrepõem-se o suficiente para "ver" a mesma transição
            # e empatar; aí o envelope não discrimina e a AMPLITUDE decide, porque é literalmente o
            # topo que procuramos. Sem isto, alguns batimentos ficavam presos num "ombro" da subida em
            # vez do pico real, sempre por empate exato do envelope entre dois pontos muito próximos.
            if env_p > env_last * 1.15 or (env_p > env_last / 1.15 and win_ecg[p] > win_ecg[r_peaks[-1]]):
                r_peaks[-1] = p
        else:
            r_peaks.append(int(p))
    r_peaks = np.array(r_peaks)

    # Onda T isolada detetada como R "solteiro" — quando o intervalo T-R (~QT+isoelétrico) cai
    # a >=0.4s da R verdadeira, ESCAPA ao desempate acima (que só compara candidatos <0.4s entre
    # si) e sobra como um "R" à parte. Rejeita só quando amplitude E envelope estão AMBOS muito
    # abaixo da mediana da própria janela — uma onda T mesmo alta nunca é afiada (T é lenta,
    # envelope baixo), mas um batimento genuíno de QRS largo/aberrante (ex.: PVC) mantém a
    # amplitude preservada mesmo com envelope mais baixo, por isso só uma condição não chega.
    # Uma onda T isolada (confirmada pelo padrão RR simétrico, que é independente) fica muito abaixo do
    # piso nas duas métricas; um batimento aberrante genuíno mantém a amplitude e sobrevive ao filtro.
    if len(r_peaks) >= 3:
        envs_all = np.array([_env_local(p) for p in r_peaks])
        amps_all = win_ecg[r_peaks]
        med_env, med_amp = np.median(envs_all), np.median(amps_all)
        if med_env > 0 and med_amp > 0:
            espurio = (envs_all < 0.15 * med_env) & (amps_all < 0.40 * med_amp)
            r_peaks = r_peaks[~espurio]

    if len(r_peaks) < 8:
        return None

    # Piso de amplitude mínima do R — o threshold relativo acima sempre encontra um "top 15%"
    # mesmo em ruído puro, porque é relativo ao próprio sinal. Um R genuíno (highpass 0.5Hz) tem
    # tipicamente amplitude bem acima de 0.1; abaixo de ~0.08 já não se distingue do ruído de
    # fundo, mesmo que a forma/distância pareçam plausíveis.
    amp_r_mediana = np.median(win_ecg[r_peaks])
    if amp_r_mediana < 0.08:
        return None

    # Segunda camada: a deteção híbrida encontra sempre "picos" mesmo em ruído puro, porque o
    # threshold é relativo ao máximo da janela — em ruído esses picos podem empurrar
    # amp_r_mediana para pouco acima do piso absoluto. A irregularidade do RR é o sinal mais
    # fiável de que não é um ritmo cardíaco real: batimentos genuínos têm RR consistente (mesmo
    # com arritmia leve, o CV é baixo); o ruído puro dá um CV muito maior. Só rejeita se
    # amplitude AINDA modesta (<0.15) E RR muito irregular — amplitude claramente forte não é
    # descartada só por variabilidade de RR mais alta.
    rr = np.diff(r_peaks) / fs
    cv_rr = np.std(rr) / np.mean(rr)
    if amp_r_mediana < 0.15 and cv_rr > 0.25:
        return None

    # Piso contra excursões anómalas na janela (artefacto de movimento/elétrodo — não é glitch de
    # 1 amostra, dura dezenas de ms, por isso `remover_picos_isolados` não o apanha): se algum
    # ponto da janela filtrada ultrapassa 4x a amplitude R típica DESTA janela, não é fisiológico.
    # Relativo à própria janela, não um valor fixo, porque a amplitude varia por registo/ganho.
    if np.max(np.abs(win_ecg)) > 4 * amp_r_mediana:
        return None

    prom_min = (np.max(win_ppg_pos) - np.min(win_ppg_pos)) * 0.15
    ppg_peaks_all, _ = find_peaks(win_ppg_pos, distance=int(fs * 0.30), prominence=prom_min)  # picos sistólicos globais
    rho_blood = 1060
    beats = []
    num_r = len(r_peaks)

    for i in range(num_r):
        r_now  = int(r_peaks[i])
        r_next = int(r_peaks[i + 1]) if i + 1 < num_r else None

        # --- Fiduciais ECG (Teste 1: argmin/argmax em janelas fixas) ---
        q_start = max(0, r_now - int(0.06 * fs))
        q_idx = int(q_start + np.argmin(win_ecg[q_start:r_now])) if q_start < r_now else r_now

        s_end_local = min(r_now + int(0.08 * fs), len(win_ecg))
        s_idx = int(r_now + np.argmin(win_ecg[r_now:s_end_local])) if r_now < s_end_local else r_now

        t_idx = None
        if r_next is not None:
            # margem antes do r_next (igual à do qn_idx) para o T nunca invadir a subida do QRS seguinte
            t_start = r_now + int(0.12 * fs)
            t_end = min(r_now + int(0.45 * fs), r_next - int(0.08 * fs))
            if t_start < t_end and t_start < len(win_ecg):
                t_idx = int(t_start + np.argmax(win_ecg[t_start:t_end]))

        qn_idx = None
        if r_next is not None:
            qn_start = max(r_now, r_next - int(0.06 * fs))
            if qn_start < r_next and qn_start < len(win_ecg):
                qn_idx = int(qn_start + np.argmin(win_ecg[qn_start:r_next]))

        # --- QRS ---
        qrs_duration = qrs_amplitude = qrs_area = r_amplitude = None
        qrs_ok = False
        if q_idx < s_idx < len(win_ecg):
            qrs_duration  = round((s_idx - q_idx) / fs, 4)
            qrs_ok        = 0.04 < qrs_duration < 0.16
            r_amplitude   = round(float(win_ecg[r_now]), 4)
            qrs_amplitude = round(float(win_ecg[r_now] - min(win_ecg[q_idx], win_ecg[s_idx])), 4)
            seg = win_ecg[q_idx:s_idx + 1] - win_ecg[q_idx]
            qrs_area = round(float(simpson(y=seg)), 4) if len(seg) > 1 else 0.0

        # --- QT / TQ / QTc / SDI ---
        qt_interval = tq_interval = qtc_interval = sdi_interaction = None
        if t_idx is not None and q_idx < t_idx < len(win_ecg):
            qt_interval = round((t_idx - q_idx) / fs, 4)
            if r_next is not None:
                rr_sec = (r_next - r_now) / fs
                qtc_interval = round(qt_interval / np.sqrt(rr_sec + 1e-6), 4)
        if t_idx is not None and qn_idx is not None and t_idx < qn_idx < len(win_ecg):
            tq_interval = round((qn_idx - t_idx) / fs, 4)
        if qt_interval is not None and tq_interval is not None:
            sdi_interaction = round(min(qt_interval / (tq_interval + 1e-6), 10.0), 4)

        # --- HR ---
        hr = round((60 * fs) / (r_next - r_now), 2) if r_next is not None else None

        # --- Pico sistólico PPG: 1º pico global depois do R (robusto a PAT curto/longo) ---
        lim = r_next if r_next is not None else min(r_now + int(0.9 * fs), len(win_ppg_pos))
        cand = ppg_peaks_all[(ppg_peaks_all > r_now) & (ppg_peaks_all <= lim)]
        ppg_pk = int(cand[0]) if len(cand) > 0 else None

        if ppg_pk is None:
            beats.append(beat_vazio(r_now, q_idx, s_idx, t_idx, qn_idx, qrs_ok, qrs_duration, r_amplitude,
                                     qrs_amplitude, qrs_area, qt_interval, tq_interval, qtc_interval, sdi_interaction, hr))
            continue

        pat = round((ppg_pk - r_now) / fs, 4)

        # --- Vale/pé PPG: mínimo ANTES do pico, limitado pelo pico PPG anterior (nunca pelo R) ---
        # Usar r_now como fronteira de início falha nos dois sentidos: com PAT curto, r_now cai
        # DEPOIS do pé verdadeiro (já na subida) e a busca nunca o alcança; com PAT grande
        # (latência do aparelho — ver janela de PAT adaptativa abaixo), r_now cai cedo demais e a
        # busca arrisca apanhar o vale de um ciclo anterior. O pé pertence à estrutura do PRÓPRIO PPG.
        picos_antes = ppg_peaks_all[ppg_peaks_all < ppg_pk]
        pe_ini = max(int(picos_antes[-1]) if len(picos_antes) > 0 else 0, ppg_pk - int(0.4 * fs), 0)
        ppg_vly = pe_ini + int(np.argmin(win_ppg_pos[pe_ini:ppg_pk])) if pe_ini < ppg_pk else max(0, ppg_pk - int(0.2 * fs))

        ppg_valley_intensity = float(win_ppg_pos[ppg_vly])
        ppg_ac_amplitude     = float(win_ppg_pos[ppg_pk] - win_ppg_pos[ppg_vly])
        omega     = (hr / 60.0) if hr is not None else 1.2
        alpha     = float(ppg_valley_intensity * np.sqrt((rho_blood * omega) / (ppg_ac_amplitude + 1e-6)))
        upslope   = win_ppg_pos[ppg_vly:ppg_pk + 1]
        deriv_ppg = np.diff(upslope) * fs
        max_slope = float(np.max(deriv_ppg)) if len(deriv_ppg) > 0 else 0.0
        b_coef    = 1.0 / (max_slope + 1e-6)
        alpha_b   = float(ppg_valley_intensity * np.sqrt((omega * rho_blood) / (b_coef + 1e-6)))

        crest_time = round((ppg_pk - ppg_vly) / fs, 4)

        # --- Fim do pulso PPG (Teste 1: primeiro mínimo local após o pico) ---
        # Limita a busca pelo PRÓXIMO PICO PPG (não pelo próximo R): usar r_next assumia PAT
        # curto, mas o PAT observado neste dataset pode ocupar quase todo o RR (latência do
        # aparelho de oximetria — ver janela de PAT adaptativa abaixo), o que deixava ppg_pk muito
        # perto de r_next e esganava esta busca (entalhe a colapsar em cima do próprio pico).
        prox_pico_ppg = ppg_peaks_all[ppg_peaks_all > ppg_pk]
        limite_pulso = int(prox_pico_ppg[0]) if len(prox_pico_ppg) > 0 else min(ppg_pk + int(0.9 * fs), len(win_ppg_pos))
        pe_start = ppg_pk + int(0.04 * fs)
        pe_end   = min(ppg_pk + int(0.50 * fs), len(win_ppg_pos), limite_pulso)
        if pe_start < pe_end:
            seg_pe = win_ppg_pos[pe_start:pe_end]
            mins, _ = find_peaks(-seg_pe, distance=int(fs * 0.05))
            pe = int(pe_start + mins[0]) if len(mins) > 0 else int(pe_start + np.argmin(seg_pe))
            decay_time = round((pe - ppg_pk) / fs, 4)
        else:
            pe = ppg_pk
            decay_time = None

        # --- Geometria do pulso ---
        p_low, p_high = max(0, ppg_pk - int(0.1 * fs)), min(len(win_ppg_pos), ppg_pk + int(0.3 * fs))
        pulso = win_ppg_pos[p_low:p_high]
        pulse_area = w50 = None
        if len(pulso) > 1 and (np.max(pulso) - np.min(pulso)) > 0.01:
            norm_p = (pulso - np.min(pulso)) / (np.max(pulso) - np.min(pulso))
            pulse_area = round(float(simpson(y=norm_p)), 4)
            idx50 = np.where(norm_p >= 0.5)[0]
            if len(idx50) > 1:
                w50 = round(float((idx50[-1] - idx50[0]) / fs), 4)

        # pat_kept definido depois do loop (janela de PAT adaptativa) — placeholder aqui
        pat_kept = False

        beats.append({
            'r_idx': r_now, 'r_next_idx': int(pe), 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
            'ppg_pk_idx': ppg_pk, 'ppg_vly_idx': ppg_vly, 'pat': pat, 'hr': hr,
            'pat_kept': pat_kept, 'qrs_ok': qrs_ok, 'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude,
            'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area, 'qt_interval': qt_interval, 'tq_interval': tq_interval,
            'qtc_interval': qtc_interval, 'sdi_interaction': sdi_interaction, 'crest_time': crest_time, 'decay_time': decay_time,
            'alpha': round(alpha, 4), 'alpha_b': round(alpha_b, 4), 'pulse_area': pulse_area, 'w50': w50,
            'ppg_valley_intensity': round(ppg_valley_intensity, 4), 'ppg_ac_amplitude': round(ppg_ac_amplitude, 4)
        })

    # --- Janela de PAT adaptativa por registo (substitui o intervalo fisiológico fixo 0.19-0.50s) ---
    # O PLETH tem latência de processamento própria do aparelho (média de vários batimentos p/
    # estabilizar o SpO2), consistente DENTRO do mesmo registo mas muito variável ENTRE registos —
    # confirmado por cross-check com o atraso R->ABP (medição direta, sem processamento de oxímetro),
    # que é apertado e estável, ao contrário do PAT do PPG cru. Em vez de um intervalo global fixo, aceitamos o agrupamento dominante e apertado
    # DESTE registo — seja qual for o valor absoluto — em bins de 40ms.
    pats_hr_ok = np.array([
        b['pat'] for b in beats
        if b['pat'] is not None and b['hr'] is not None and 45 < b['hr'] < 140
    ])
    if len(pats_hr_ok) >= 6:
        bins = np.arange(0, pats_hr_ok.max() + 0.04, 0.04)
        hist, edges = np.histogram(pats_hr_ok, bins=bins)
        i_max = int(np.argmax(hist))
        cl_min, cl_max = edges[i_max] - 0.04, edges[i_max + 1] + 0.04
        for b in beats:
            if b['pat'] is not None and b['hr'] is not None and 45 < b['hr'] < 140:
                b['pat_kept'] = bool(cl_min <= b['pat'] <= cl_max)

    # --- Filtragem IQR sobre os PATs aceites (Teste 1: remove outliers batimento a batimento) ---
    pats = np.array([b['pat'] for b in beats if b['pat'] is not None and b['pat_kept']])
    if len(pats) >= 4:
        q1, q3 = np.percentile(pats, [25, 75])
        iqr = (q3 - q1) if (q3 - q1) != 0 else 0.05
        lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        for b in beats:
            if b['pat'] is not None and b['pat_kept']:
                b['pat_kept'] = bool(lim_inf <= b['pat'] <= lim_sup)

    kept = [b for b in beats if b['pat_kept'] and b['pat'] is not None]
    kept_pats = np.array([b['pat'] for b in kept])
    # Teste 1 exige >= 6 batimentos limpos E baixa dispersão do PAT
    if len(kept) < 6 or np.std(kept_pats) >= 0.05:
        return beats, None, win_ecg, win_ppg

    def med(lst, key):
        vals = [b[key] for b in lst if b[key] is not None]
        return round(float(np.mean(vals)), 4) if vals else None

    features = {
        'pat_mean': med(kept, 'pat'), 'hr_mean': med(kept, 'hr'),
        'qrs_duration_mean': med([b for b in kept if b['qrs_ok']], 'qrs_duration'),
        'r_amplitude_mean':  med([b for b in kept if b['qrs_ok']], 'r_amplitude'),
        'qrs_amplitude_mean':med([b for b in kept if b['qrs_ok']], 'qrs_amplitude'),
        'qrs_area_mean':     med([b for b in kept if b['qrs_ok']], 'qrs_area'),
        'qt_interval_mean':  med([b for b in kept if b['qt_interval']  is not None], 'qt_interval'),
        'tq_interval_mean':  med([b for b in kept if b['tq_interval']  is not None], 'tq_interval'),
        'qtc_interval_mean': med([b for b in kept if b['qtc_interval'] is not None], 'qtc_interval'),
        'sdi_interaction_mean': med([b for b in kept if b['sdi_interaction'] is not None], 'sdi_interaction'),
        'crest_time_mean':   med(kept, 'crest_time'), 'decay_time_mean': med(kept, 'decay_time'),
        'alpha_mean':        med(kept, 'alpha'), 'alpha_b_mean': med(kept, 'alpha_b'),
        'pulse_area_mean':   med([b for b in kept if b['pulse_area'] is not None], 'pulse_area'),
        'w50_mean':          med([b for b in kept if b['pulse_area'] is not None], 'w50'),
        'ppg_valley_intensity_mean': med(kept, 'ppg_valley_intensity'),
        'ppg_ac_amplitude_mean':     med(kept, 'ppg_ac_amplitude'),
    }
    return beats, features, win_ecg, win_ppg


def processar_linha_individual(row):
    """
    Função atómica que processa apenas UMA linha do CSV original.
    Distribuída em paralelo por múltiplos cores.
    """
    s_id = str(row['subject_folder'])

    # --- DETEÇÃO AUTOMÁTICA DE DATASET ---
    if len(s_id) == 7:
        dataset_label = "MIMIC-III"
        canais_abp, canais_ppg = ['ABP', 'ART'], ['PLETH', 'PPG']
    elif len(s_id) == 9:
        dataset_label = "MIMIC-IV"
        canais_abp, canais_ppg = ['ABP'], ['PLETH']
    else:
        return []

    seg_name = str(row['segment'])
    try:
        seg_order = int(seg_name.split('_')[-1])
    except:
        seg_order = 0

    base_path = row['absolute_path'].replace('.dat', '').replace('.hea', '')
    if not os.path.exists(base_path + '.hea'):
        return []

    try:
        # MEMÓRIA: ler primeiro só o HEADER (barato) para descobrir os canais, e depois carregar
        # APENAS os 3 canais necessários (ECG/PPG/ABP) em vez do registo inteiro. Alguns registos
        # têm 6-8 canais e até 75h de duração (>1.5 GB carregados por inteiro em float64); carregar
        # só 3 canais + converter para float32 reduz o pico de RAM por worker para <1/4. Sem isto,
        # vários workers a apanhar registos grandes ao mesmo tempo esgotavam os 30 GB (OOM/crash).
        hdr = wfdb.rdheader(base_path)
        fs = hdr.fs
        sig_names = [s.upper() for s in hdr.sig_name]

        idx_ecg = selecionar_ecg(sig_names)
        idx_ppg = next((i for i, s in enumerate(sig_names) if any(x in s for x in canais_ppg)), None)
        idx_abp = next((i for i, s in enumerate(sig_names) if any(x in s for x in canais_abp)), None)
        if idx_ecg is None or idx_ppg is None or idx_abp is None:
            return []

        record = wfdb.rdrecord(base_path, channels=[idx_ecg, idx_ppg, idx_abp])
        ecg_raw = record.p_signal[:, 0].astype(np.float32)
        ppg_raw = record.p_signal[:, 1].astype(np.float32)
        abp_raw = record.p_signal[:, 2].astype(np.float32)
        del record  # liberta a matriz float64 devolvida pelo wfdb assim que temos as cópias float32

        if np.isnan(ecg_raw).any() or np.isnan(ppg_raw).any() or np.isnan(abp_raw).any():
            ecg_raw = pd.Series(ecg_raw).interpolate().ffill().bfill().values
            ppg_raw = pd.Series(ppg_raw).interpolate().ffill().bfill().values
            abp_raw = pd.Series(abp_raw).interpolate().ffill().bfill().values

        ecg_raw = remover_picos_isolados(ecg_raw)  # 🆕 repara glitches de 1 amostra (dropout do ADC) antes do highpass

        # Filtragem Biomédica dos Sinais
        ppg_filt = butter_lowpass_filter(ppg_raw, 8, fs)
        ecg_filt = butter_highpass_filter(ecg_raw, 0.5, fs)
        # Correção de inversão: se skew(ECG)<0 o sinal está invertido → repõe R para cima
        if skew(ecg_filt) < 0:
            ecg_filt = -ecg_filt
    except:
        return []

    resultados_da_linha = []

    # --- Janelas deslizantes com sobreposição ---
    samples_per_win = int(WINDOW_SIZE_SEC * fs)
    step_samples = int(STRIDE_SEC * fs)
    num_windows = int((len(ecg_filt) - samples_per_win) / step_samples) + 1

    for w in range(num_windows):
        start = w * step_samples
        end = start + samples_per_win

        win_ecg, win_ppg, win_abp = ecg_filt[start:end], ppg_filt[start:end], abp_raw[start:end]

        if np.std(win_ppg) < 0.005 or (np.max(win_abp) - np.min(win_abp)) < 15:
            continue

        # Gate ABP (targets fisiológicos) — computado cedo, antes do processamento pesado de batimentos
        sbp, dbp, sbp_std, pp = extrair_targets_abp_robusto(win_abp, fs)
        if sbp is None or not (pp > 25 and sbp_std < 12 and 65 < sbp < 195 and 35 < dbp < 115):
            continue

        res = processar_janela_teste1(win_ecg, win_ppg, fs)
        if res is None or res[1] is None:
            continue

        _, features, _, _ = res
        resultados_da_linha.append({
            'subject_id': s_id,
            'segment_name': seg_name,
            'segment_order': seg_order,
            'window_idx': w,
            'dataset': dataset_label,
            'gender': row['gender'],
            'age': row['age'],
            'pat_mean': features['pat_mean'],
            'hr_mean': features['hr_mean'],
            'qrs_duration': features['qrs_duration_mean'],
            'r_amplitude': features['r_amplitude_mean'],
            'qrs_amplitude': features['qrs_amplitude_mean'],
            'qrs_area': features['qrs_area_mean'],
            'w50': features['w50_mean'],
            'area_mean': features['pulse_area_mean'],
            'ppg_valley_intensity': features['ppg_valley_intensity_mean'],
            'ppg_ac_amplitude': features['ppg_ac_amplitude_mean'],
            'ppg_crest_time': features['crest_time_mean'],
            'ppg_decay_time': features['decay_time_mean'],
            'alpha_womersley': features['alpha_mean'],
            'alpha_womersley_b': features['alpha_b_mean'],
            'qt_interval': features['qt_interval_mean'],
            'tq_interval': features['tq_interval_mean'],
            'qtc_interval': features['qtc_interval_mean'],
            'sdi_interaction': features['sdi_interaction_mean'],
            'sbp_target': round(sbp, 2),
            'dbp_target': round(dbp, 2)
        })

    return resultados_da_linha

def processar_datasets_hibridos():
    for csv_file in INPUT_CSVS:
        if not os.path.exists(csv_file):
            print(f"Aviso: {csv_file} não encontrado.")
            continue

        df = pd.read_csv(csv_file)
        print(f"A inicializar processamento paralelo para {len(df)} registos em: {csv_file}")

        num_cores = N_JOBS if N_JOBS is not None else multiprocessing.cpu_count()
        print(f"Motor de Processamento Paralelo ativado utilizando {num_cores} CORES (de {multiprocessing.cpu_count()} disponíveis).")

        # max_nbytes=None evita que o joblib crie memmaps temporários em disco para os argumentos;
        # aqui os argumentos (uma linha do CSV) são pequenos, o que pesa é o que cada worker carrega.
        raw_results = Parallel(n_jobs=num_cores, backend="multiprocessing", verbose=10, max_nbytes=None)(
            delayed(processar_linha_individual)(row) for _, row in df.iterrows()
        )

        features_list = []
        for res in raw_results:
            if res:
                features_list.extend(res)

        df_feat = pd.DataFrame(features_list)
        if not df_feat.empty:
            os.makedirs(PATH_FEATURES_OUT, exist_ok=True)
            for g, label in [('M', 'homens'), ('F', 'mulheres')]:
                subset = df_feat[df_feat['gender'] == g]
                if not subset.empty:
                    # Guardar ficheiros finais segregados por género — nome claro: variante "baseline"
                    subset.to_csv(f"{PATH_FEATURES_OUT}/features_{label}_baseline.csv", index=False)
            print(f"SUCESSO PARALELO! Janelas totais processadas: {len(df_feat)}")
        else:
            print("Nenhuma janela válida gerada nesta execução.")

if __name__ == "__main__":
    processar_datasets_hibridos()


### 1.2 Visualização

Visualização dos sinais e deteções do método Teste 1 (partilhada pelo baseline e pela variante com SQI).

In [ ]:
import os, json
import pandas as pd
import wfdb
import numpy as np
from scipy.signal import find_peaks, butter, filtfilt
from scipy.integrate import simpson
from scipy.stats import skew

# ========== CONFIGURAÇÃO ==========
PATH_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"
INPUT_CSV = f"{PATH_DATASET}/dataset_para_extração_features.csv"

USE_SQI_GATE = False

if USE_SQI_GATE:
    OUTPUT_JSON = "/home/filipe/Desktop/Tese/algoritmo-ml/Visualização_e_Comparação/visualizacao_sinais_com_sqi.json"
else:
    OUTPUT_JSON = "/home/filipe/Desktop/Tese/algoritmo-ml/Visualização_e_Comparação/visualizacao_sinais_baseline.json"

WINDOW_SIZE_SEC = 15
STRIDE_SEC = 10
MAX_PACIENTES = 10
MAX_JANELAS_POR_PACIENTE = 150
PACIENTES_ESPECIFICOS = None

# --- SQI de skewness (Teste 1 aprimorado) — só para visualizar/filtrar as janelas ---
# USE_SQI_GATE=False -> mostra o Teste 1 puro (todas as janelas que passam nos gates originais).
# USE_SQI_GATE=True  -> aplica a MESMA retenção adaptativa por paciente da secção 2.1 (não um
#                        corte cego): mantém sqi_skew>=SQI_SKEW_MIN, mas nunca abaixo do piso
#                        mínimo por paciente — ver FRACAO_MINIMA_RETIDA/MIN_JANELAS_ABS abaixo.
# Em qualquer caso o valor do SQI é exportado por janela (campo 'sqi_skew') para o veres no gráfico.
SQI_SKEW_MIN = 0.0
# 🆕 Rede de segurança por paciente — MESMOS valores/lógica da secção 2.1 (ver lá a justificação
# empírica). Mantidos em sincronia manualmente: se mudares um, muda o outro.
FRACAO_MINIMA_RETIDA = 0.5
MIN_JANELAS_ABS = 20


# ========== FILTROS (Teste 1) ==========
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return filtfilt(b, a, data)

def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='high', analog=False)
    return filtfilt(b, a, data)

def butter_bandpass_filter(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype='band', analog=False)
    return filtfilt(b, a, data)

def sqi_skewness(seg):
    if seg is None or len(seg) < 10 or np.std(seg) < 1e-9:
        return -np.inf
    return float(skew(seg))

def limpar_nan(x):
    if np.isnan(x).any():
        return pd.Series(x).interpolate().ffill().bfill().values
    return x

def remover_picos_isolados(sinal, limiar=0.15):
    """Repara glitches de UMA amostra (ex: dropout do ADC): marca a amostra i só se AMBOS os
    vizinhos imediatos divergem muito dela mas concordam ENTRE si — um QRS genuíno nunca tem essa
    assinatura (a subida ocupa várias amostras, vizinhos não concordam entre si). Substitui pela
    média dos vizinhos. Um filtro de mediana/MAD com janela maior corta os próprios picos R."""
    s = np.asarray(sinal, dtype=float)
    if len(s) < 3:
        return s
    d_ant = np.abs(s[1:-1] - s[:-2])
    d_prox = np.abs(s[1:-1] - s[2:])
    d_vizinhos = np.abs(s[:-2] - s[2:])
    glitch = (d_ant > limiar) & (d_prox > limiar) & (d_vizinhos < limiar * 0.5)
    limpo = s.copy()
    idx = np.where(glitch)[0] + 1
    limpo[idx] = (s[idx - 1] + s[idx + 1]) / 2
    return limpo


# ========== CONVERSÃO PARA TIPOS NATIVOS ==========
def to_native(obj):
    if isinstance(obj, (np.integer,)):   return int(obj)
    if isinstance(obj, (np.floating,)):  return float(obj)
    if isinstance(obj, np.ndarray):      return obj.tolist()
    if isinstance(obj, dict):            return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):   return [to_native(i) for i in obj]
    return obj


# ========== ABP MARKS ==========
def extrair_abp_markers(win_abp, fs):
    sbp_idxs, _ = find_peaks(win_abp,  distance=int(fs * 0.4), height=np.percentile(win_abp, 50))
    dbp_idxs, _ = find_peaks(-win_abp, distance=int(fs * 0.4))
    return [int(i) for i in sbp_idxs], [int(i) for i in dbp_idxs]

def extrair_abp_targets(win_abp, fs):
    sbp_idxs, dbp_idxs = extrair_abp_markers(win_abp, fs)
    if len(sbp_idxs) < 3 or len(dbp_idxs) < 3:
        return None, None, None, None
    sbp_vals = win_abp[sbp_idxs]
    dbp_vals = win_abp[dbp_idxs]
    sbp = float(np.mean(sbp_vals))
    dbp = float(np.mean(dbp_vals))
    sbp_std = float(np.std(sbp_vals))
    pp = sbp - dbp
    return sbp, dbp, sbp_std, pp


# ========== BEAT VAZIO ==========
def beat_vazio(r_now, q_idx, s_idx, t_idx, qn_idx, qrs_ok, qrs_duration, r_amplitude,
               qrs_amplitude, qrs_area, qt_interval, tq_interval, qtc_interval, sdi_interaction, hr):
    """Batimento sem pico PPG correspondente (ppg_pk é None). As métricas do lado ECG (Q/R/S/T,
    duração/amplitude do QRS, HR) já foram calculadas antes deste ponto e são independentes de
    haver pulso PPG a seguir — preservá-las evita marcar um QRS válido como 'excluído' só porque
    o PPG falhou (ex.: pulso fora da janela, ruído momentâneo no PPG)."""
    return {
        'r_idx': int(r_now), 'r_next_idx': None, 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
        'ppg_pk_idx': None, 'ppg_vly_idx': None, 'pat': None, 'hr': hr, 'pat_kept': False, 'qrs_ok': qrs_ok,
        'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude, 'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area,
        'qt_interval': qt_interval, 'tq_interval': tq_interval, 'qtc_interval': qtc_interval, 'sdi_interaction': sdi_interaction,
        'crest_time': None, 'decay_time': None, 'alpha': None, 'alpha_b': None, 'pulse_area': None, 'w50': None,
        'ppg_valley_intensity': None, 'ppg_ac_amplitude': None
    }


# ========== PIPELINE TESTE 1 (SEM NEUROKIT) ==========
def processar_janela_teste1(win_ecg, win_ppg, fs):
    """Processa UMA janela com a lógica EXATA do Teste 1 (scipy, sem NeuroKit).
        win_ecg : ECG já filtrado (highpass 0.5 Hz, remove baseline wander)
        win_ppg : PPG já filtrado (lowpass 8 Hz)
    Devolve (beats, features, ecg_plot, ppg_plot) no MESMO formato do Teste 2 viz, para
    o mesmo front-end poder desenhar. Os índices dos batimentos são relativos à janela.
    Nota: 'r_next_idx' guarda o fim do pulso PPG (pe) — mesma convenção do Teste 2 viz."""
    win_ppg_pos = win_ppg - np.min(win_ppg) + 0.01   # positividade p/ alpha (não move os índices)

    # Deteção de picos R — abordagem HÍBRIDA (amplitude do ECG + envelope de Pan-Tompkins):
    #  • Candidatos por AMPLITUDE no próprio ECG (picos positivos ≥15% do máximo). Isto apanha
    #    de forma fiável todos os R genuínos, mesmo os de QRS menos afiado — um detetor puramente
    #    baseado no envelope (threshold global 0.3*max) perdia R verdadeiros quando o máximo do
    #    envelope estava inflado por uma onda S muito abrupta noutro ponto da janela.
    #  • O envelope (derivada²+integração ~80ms, realça a subida/descida rápida do QRS e suprime a
    #    onda T, mais lenta) serve para DESEMPATAR: quando dois candidatos ficam a <0.4s, fica o de
    #    maior envelope local (o R, sempre mais afiado) — mesmo que a onda T seja MAIS ALTA que o R
    #    (R atenuado). Assim resolve-se o problema clássico de marcar a T como R sem usar amplitude.
    deriv = np.diff(win_ecg, prepend=win_ecg[0]) * fs
    win_integ = max(1, int(0.08 * fs))
    envelope = np.convolve(deriv ** 2, np.ones(win_integ) / win_integ, mode='same')

    def _env_local(p):
        lo, hi = max(0, p - int(0.05 * fs)), min(len(envelope), p + int(0.05 * fs))
        return np.max(envelope[lo:hi])

    candidatos, _ = find_peaks(win_ecg, height=0.15 * np.max(win_ecg))
    r_peaks = []
    for p in candidatos:
        if r_peaks and (p - r_peaks[-1]) < int(fs * 0.4):
            env_p, env_last = _env_local(p), _env_local(r_peaks[-1])
            # Desempate: só troca por ENVELOPE quando a diferença é clara (>15%) — é o caso T-vs-R
            # (uma T mesmo alta tem sempre envelope MUITO menor, tipicamente >10x, porque é lenta).
            # Quando os envelopes ficam próximos, os dois candidatos pertencem à MESMA subida aguda
            # do QRS (ex.: um "ombro"/ruído no meio da subida antes do topo verdadeiro) — as janelas
            # de +-50ms usadas por _env_local sobrepõem-se o suficiente para "ver" a mesma transição
            # e empatar; aí o envelope não discrimina e a AMPLITUDE decide, porque é literalmente o
            # topo que procuramos. Sem isto, alguns batimentos ficavam presos num "ombro" da subida em
            # vez do pico real, sempre por empate exato do envelope entre dois pontos muito próximos.
            if env_p > env_last * 1.15 or (env_p > env_last / 1.15 and win_ecg[p] > win_ecg[r_peaks[-1]]):
                r_peaks[-1] = p
        else:
            r_peaks.append(int(p))
    r_peaks = np.array(r_peaks)

    # Onda T isolada detetada como R "solteiro" — quando o intervalo T-R (~QT+isoelétrico) cai
    # a >=0.4s da R verdadeira, ESCAPA ao desempate acima (que só compara candidatos <0.4s entre
    # si) e sobra como um "R" à parte. Rejeita só quando amplitude E envelope estão AMBOS muito
    # abaixo da mediana da própria janela — uma onda T mesmo alta nunca é afiada (T é lenta,
    # envelope baixo), mas um batimento genuíno de QRS largo/aberrante (ex.: PVC) mantém a
    # amplitude preservada mesmo com envelope mais baixo, por isso só uma condição não chega.
    # Uma onda T isolada (confirmada pelo padrão RR simétrico, que é independente) fica muito abaixo do
    # piso nas duas métricas; um batimento aberrante genuíno mantém a amplitude e sobrevive ao filtro.
    if len(r_peaks) >= 3:
        envs_all = np.array([_env_local(p) for p in r_peaks])
        amps_all = win_ecg[r_peaks]
        med_env, med_amp = np.median(envs_all), np.median(amps_all)
        if med_env > 0 and med_amp > 0:
            espurio = (envs_all < 0.15 * med_env) & (amps_all < 0.40 * med_amp)
            r_peaks = r_peaks[~espurio]

    if len(r_peaks) < 8:
        return None

    # Piso de amplitude mínima do R — o threshold relativo (0.3*max) acima sempre encontra um
    # "top 30%" mesmo em ruído puro, porque é relativo ao próprio sinal. Um R genuíno (highpass
    # 0.5Hz) tem tipicamente amplitude bem acima de 0.1; abaixo de ~0.08 já não se distingue do
    # ruído de fundo, mesmo que a forma/distância pareçam plausíveis (validado: janelas de ECG
    # ruidoso sem sinal real medem ~0.01-0.05, contra >=0.07-0.1 em janelas com sinal genuíno).
    amp_r_mediana = np.median(win_ecg[r_peaks])
    if amp_r_mediana < 0.08:
        return None

    # Segunda camada: a deteção híbrida (candidatos por amplitude do próprio ECG) encontra sempre
    # "picos" mesmo em ruído puro, porque o threshold é relativo ao máximo da janela — em ruído
    # esses picos podem empurrar amp_r_mediana para pouco acima do piso absoluto. A irregularidade
    # do RR é o sinal mais fiável de que não é um ritmo cardíaco real: batimentos genuínos têm
    # RR consistente (mesmo com arritmia leve, o CV é baixo); o ruído puro dá um CV muito maior. Só rejeita se amplitude AINDA modesta (<0.15) E RR muito irregular — uma janela com
    # amplitude claramente forte não é descartada só por variabilidade de RR mais alta.
    rr = np.diff(r_peaks) / fs
    cv_rr = np.std(rr) / np.mean(rr)
    if amp_r_mediana < 0.15 and cv_rr > 0.25:
        return None

    # Piso contra excursões anómalas na janela (artefacto de movimento/elétrodo — não é glitch de
    # 1 amostra, dura dezenas de ms, por isso `remover_picos_isolados` não o apanha): se algum
    # ponto da janela filtrada ultrapassa 4x a amplitude R típica DESTA janela, não é fisiológico
    # (o limiar fica entre o que as janelas boas atingem e o que um artefacto real atinge). Relativo à própria janela, não um valor fixo, porque a amplitude varia por
    # registo/ganho.
    if np.max(np.abs(win_ecg)) > 4 * amp_r_mediana:
        return None

    prom_min = (np.max(win_ppg_pos) - np.min(win_ppg_pos)) * 0.15
    ppg_peaks_all, _ = find_peaks(win_ppg_pos, distance=int(fs * 0.30), prominence=prom_min)  # picos sistólicos globais
    rho_blood = 1060
    beats = []
    num_r = len(r_peaks)

    for i in range(num_r):
        r_now  = int(r_peaks[i])
        r_next = int(r_peaks[i + 1]) if i + 1 < num_r else None

        # --- Fiduciais ECG (Teste 1: argmin/argmax em janelas fixas) ---
        q_start = max(0, r_now - int(0.06 * fs))
        q_idx = int(q_start + np.argmin(win_ecg[q_start:r_now])) if q_start < r_now else r_now

        s_end_local = min(r_now + int(0.08 * fs), len(win_ecg))
        s_idx = int(r_now + np.argmin(win_ecg[r_now:s_end_local])) if r_now < s_end_local else r_now

        t_idx = None
        if r_next is not None:
            # margem antes do r_next (igual à do qn_idx) para o T nunca invadir a subida do QRS seguinte
            t_start = r_now + int(0.12 * fs)
            t_end = min(r_now + int(0.45 * fs), r_next - int(0.08 * fs))
            if t_start < t_end and t_start < len(win_ecg):
                t_idx = int(t_start + np.argmax(win_ecg[t_start:t_end]))

        qn_idx = None
        if r_next is not None:
            qn_start = max(r_now, r_next - int(0.06 * fs))
            if qn_start < r_next and qn_start < len(win_ecg):
                qn_idx = int(qn_start + np.argmin(win_ecg[qn_start:r_next]))

        # --- QRS ---
        qrs_duration = qrs_amplitude = qrs_area = r_amplitude = None
        qrs_ok = False
        if q_idx < s_idx < len(win_ecg):
            qrs_duration  = round((s_idx - q_idx) / fs, 4)
            qrs_ok        = 0.04 < qrs_duration < 0.16
            r_amplitude   = round(float(win_ecg[r_now]), 4)
            qrs_amplitude = round(float(win_ecg[r_now] - min(win_ecg[q_idx], win_ecg[s_idx])), 4)
            seg = win_ecg[q_idx:s_idx + 1] - win_ecg[q_idx]
            qrs_area = round(float(simpson(y=seg)), 4) if len(seg) > 1 else 0.0

        # --- QT / TQ / QTc / SDI ---
        qt_interval = tq_interval = qtc_interval = sdi_interaction = None
        if t_idx is not None and q_idx < t_idx < len(win_ecg):
            qt_interval = round((t_idx - q_idx) / fs, 4)
            if r_next is not None:
                rr_sec = (r_next - r_now) / fs
                qtc_interval = round(qt_interval / np.sqrt(rr_sec + 1e-6), 4)
        if t_idx is not None and qn_idx is not None and t_idx < qn_idx < len(win_ecg):
            tq_interval = round((qn_idx - t_idx) / fs, 4)
        if qt_interval is not None and tq_interval is not None:
            sdi_interaction = round(min(qt_interval / (tq_interval + 1e-6), 10.0), 4)

        # --- HR ---
        hr = round((60 * fs) / (r_next - r_now), 2) if r_next is not None else None

        # --- Pico sistólico PPG: 1º pico global depois do R (robusto a PAT curto/longo) ---
        lim = r_next if r_next is not None else min(r_now + int(0.9 * fs), len(win_ppg_pos))
        cand = ppg_peaks_all[(ppg_peaks_all > r_now) & (ppg_peaks_all <= lim)]
        ppg_pk = int(cand[0]) if len(cand) > 0 else None

        if ppg_pk is None:
            beats.append(beat_vazio(r_now, q_idx, s_idx, t_idx, qn_idx, qrs_ok, qrs_duration, r_amplitude,
                                     qrs_amplitude, qrs_area, qt_interval, tq_interval, qtc_interval, sdi_interaction, hr))
            continue

        pat = round((ppg_pk - r_now) / fs, 4)

        # --- Vale/pé PPG: mínimo ANTES do pico, limitado pelo pico PPG anterior (nunca pelo R) ---
        # Usar r_now como fronteira de início falha nos dois sentidos: com PAT curto, r_now cai
        # DEPOIS do pé verdadeiro (já na subida) e a busca nunca o alcança; com PAT grande
        # (latência do aparelho — ver janela de PAT adaptativa), r_now cai cedo demais e a busca
        # arrisca apanhar o vale de um ciclo anterior. O pé pertence à estrutura do PRÓPRIO PPG.
        picos_antes = ppg_peaks_all[ppg_peaks_all < ppg_pk]
        pe_ini = max(int(picos_antes[-1]) if len(picos_antes) > 0 else 0, ppg_pk - int(0.4 * fs), 0)
        ppg_vly = pe_ini + int(np.argmin(win_ppg_pos[pe_ini:ppg_pk])) if pe_ini < ppg_pk else max(0, ppg_pk - int(0.2 * fs))

        ppg_valley_intensity = float(win_ppg_pos[ppg_vly])
        ppg_ac_amplitude     = float(win_ppg_pos[ppg_pk] - win_ppg_pos[ppg_vly])
        omega     = (hr / 60.0) if hr is not None else 1.2
        alpha     = float(ppg_valley_intensity * np.sqrt((rho_blood * omega) / (ppg_ac_amplitude + 1e-6)))
        upslope   = win_ppg_pos[ppg_vly:ppg_pk + 1]
        deriv     = np.diff(upslope) * fs
        max_slope = float(np.max(deriv)) if len(deriv) > 0 else 0.0
        b_coef    = 1.0 / (max_slope + 1e-6)
        alpha_b   = float(ppg_valley_intensity * np.sqrt((omega * rho_blood) / (b_coef + 1e-6)))

        crest_time = round((ppg_pk - ppg_vly) / fs, 4)

        # --- Fim do pulso PPG (Teste 1: primeiro mínimo local após o pico) ---
        # Limita a busca pelo PRÓXIMO PICO PPG (não pelo próximo R): usar r_next assumia PAT
        # curto, mas o PAT observado neste dataset pode ocupar quase todo o RR (latência do
        # aparelho de oximetria — ver janela de PAT adaptativa acima), o que deixava ppg_pk muito
        # perto de r_next e esganava esta busca (entalhe a colapsar em cima do próprio pico).
        prox_pico_ppg = ppg_peaks_all[ppg_peaks_all > ppg_pk]
        limite_pulso = int(prox_pico_ppg[0]) if len(prox_pico_ppg) > 0 else min(ppg_pk + int(0.9 * fs), len(win_ppg_pos))
        pe_start = ppg_pk + int(0.04 * fs)
        pe_end   = min(ppg_pk + int(0.50 * fs), len(win_ppg_pos), limite_pulso)
        if pe_start < pe_end:
            seg_pe = win_ppg_pos[pe_start:pe_end]
            mins, _ = find_peaks(-seg_pe, distance=int(fs * 0.05))
            pe = int(pe_start + mins[0]) if len(mins) > 0 else int(pe_start + np.argmin(seg_pe))
            decay_time = round((pe - ppg_pk) / fs, 4)
        else:
            pe = ppg_pk
            decay_time = None

        # --- Geometria do pulso ---
        p_low, p_high = max(0, ppg_pk - int(0.1 * fs)), min(len(win_ppg_pos), ppg_pk + int(0.3 * fs))
        pulso = win_ppg_pos[p_low:p_high]
        pulse_area = w50 = None
        if len(pulso) > 1 and (np.max(pulso) - np.min(pulso)) > 0.01:
            norm_p = (pulso - np.min(pulso)) / (np.max(pulso) - np.min(pulso))
            pulse_area = round(float(simpson(y=norm_p)), 4)
            idx50 = np.where(norm_p >= 0.5)[0]
            if len(idx50) > 1:
                w50 = round(float((idx50[-1] - idx50[0]) / fs), 4)

        # pat_kept definido depois do loop (janela de PAT adaptativa) — placeholder aqui
        pat_kept = False

        beats.append({
            'r_idx': r_now, 'r_next_idx': int(pe), 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
            'ppg_pk_idx': ppg_pk, 'ppg_vly_idx': ppg_vly, 'pat': pat, 'hr': hr,
            'pat_kept': pat_kept, 'qrs_ok': qrs_ok, 'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude,
            'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area, 'qt_interval': qt_interval, 'tq_interval': tq_interval,
            'qtc_interval': qtc_interval, 'sdi_interaction': sdi_interaction, 'crest_time': crest_time, 'decay_time': decay_time,
            'alpha': round(alpha, 4), 'alpha_b': round(alpha_b, 4), 'pulse_area': pulse_area, 'w50': w50,
            'ppg_valley_intensity': round(ppg_valley_intensity, 4), 'ppg_ac_amplitude': round(ppg_ac_amplitude, 4)
        })

    # --- Janela de PAT adaptativa por registo (substitui o intervalo fisiológico fixo 0.19-0.50s) ---
    # O PLETH tem latência de processamento própria do aparelho (média de vários batimentos p/
    # estabilizar o SpO2), consistente DENTRO do mesmo registo mas muito variável ENTRE registos —
    # confirmado por cross-check com o atraso R->ABP (medição direta, sem processamento de oxímetro),
    # que é apertado e estável, ao contrário do PAT do PPG cru. Em vez de um intervalo global fixo, aceitamos o agrupamento dominante e apertado
    # DESTE registo — seja qual for o valor absoluto — em bins de 40ms.
    pats_hr_ok = np.array([
        b['pat'] for b in beats
        if b['pat'] is not None and b['hr'] is not None and 45 < b['hr'] < 140
    ])
    if len(pats_hr_ok) >= 6:
        bins = np.arange(0, pats_hr_ok.max() + 0.04, 0.04)
        hist, edges = np.histogram(pats_hr_ok, bins=bins)
        i_max = int(np.argmax(hist))
        cl_min, cl_max = edges[i_max] - 0.04, edges[i_max + 1] + 0.04
        for b in beats:
            if b['pat'] is not None and b['hr'] is not None and 45 < b['hr'] < 140:
                b['pat_kept'] = bool(cl_min <= b['pat'] <= cl_max)

    # --- Filtragem IQR sobre os PATs aceites (Teste 1: remove outliers batimento a batimento) ---
    pats = np.array([b['pat'] for b in beats if b['pat'] is not None and b['pat_kept']])
    if len(pats) >= 4:
        q1, q3 = np.percentile(pats, [25, 75])
        iqr = (q3 - q1) if (q3 - q1) != 0 else 0.05
        lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        for b in beats:
            if b['pat'] is not None and b['pat_kept']:
                b['pat_kept'] = bool(lim_inf <= b['pat'] <= lim_sup)

    kept = [b for b in beats if b['pat_kept'] and b['pat'] is not None]
    kept_pats = np.array([b['pat'] for b in kept])
    # Teste 1 exige >= 6 batimentos limpos E baixa dispersão do PAT
    if len(kept) < 6 or np.std(kept_pats) >= 0.05:
        return beats, None, win_ecg, win_ppg

    def med(lst, key):
        vals = [b[key] for b in lst if b[key] is not None]
        return round(float(np.mean(vals)), 4) if vals else None

    features = {
        'pat_mean': med(kept, 'pat'), 'hr_mean': med(kept, 'hr'),
        'qrs_duration_mean': med([b for b in kept if b['qrs_ok']], 'qrs_duration'),
        'r_amplitude_mean':  med([b for b in kept if b['qrs_ok']], 'r_amplitude'),
        'qrs_amplitude_mean':med([b for b in kept if b['qrs_ok']], 'qrs_amplitude'),
        'qrs_area_mean':     med([b for b in kept if b['qrs_ok']], 'qrs_area'),
        'qt_interval_mean':  med([b for b in kept if b['qt_interval']  is not None], 'qt_interval'),
        'tq_interval_mean':  med([b for b in kept if b['tq_interval']  is not None], 'tq_interval'),
        'qtc_interval_mean': med([b for b in kept if b['qtc_interval'] is not None], 'qtc_interval'),
        'sdi_interaction_mean': med([b for b in kept if b['sdi_interaction'] is not None], 'sdi_interaction'),
        'crest_time_mean':   med(kept, 'crest_time'), 'decay_time_mean': med(kept, 'decay_time'),
        'alpha_mean':        med(kept, 'alpha'), 'alpha_b_mean': med(kept, 'alpha_b'),
        'pulse_area_mean':   med([b for b in kept if b['pulse_area'] is not None], 'pulse_area'),
        'w50_mean':          med([b for b in kept if b['pulse_area'] is not None], 'w50'),
        'ppg_valley_intensity_mean': med(kept, 'ppg_valley_intensity'),
        'ppg_ac_amplitude_mean':     med(kept, 'ppg_ac_amplitude'),
    }
    return beats, features, win_ecg, win_ppg


# ========== LEITURA DE CANAIS (idêntico ao Teste 2 viz) ==========
# Canal de ECG: EXCLUSIVAMENTE a derivação II (nome exato, não substring — 'V'/'I' soltos
# apanhavam CVP/ICP/IC2 por conterem essas letras). Nada de fallback para outras derivações: o
# setup real do projeto (AD8232, 3 elétrodos) mede Lead II, e treinar/visualizar com a morfologia
# de QRS/T de outra derivação criaria uma discrepância treino↔uso real. Neste dataset todos os
# 9646 registos têm II, por isso esta restrição não perde nenhum paciente.
def selecionar_ecg(sig_names):
    return sig_names.index('II') if 'II' in sig_names else None

def ler_record(row):
    try:
        folder = str(row['subject_folder'])
        seg_name = str(row['segment'])
        path_record = f"{PATH_DATASET}/{folder}/{seg_name}"
        record = wfdb.rdrecord(path_record)
        sig_names = [s.upper() for s in record.sig_name]

        ppg_opts = ['PPG', 'PLETH']
        abp_opts = ['ABP', 'ART', 'PRESS']

        ecg_idx = selecionar_ecg(sig_names)
        ppg_idx = next((i for i, s in enumerate(sig_names) if any(x in s for x in ppg_opts)), None)
        abp_idx = next((i for i, s in enumerate(sig_names) if any(x in s for x in abp_opts)), None)

        if ecg_idx is None or ppg_idx is None or abp_idx is None:
            return None

        return record.p_signal[:, ecg_idx], record.p_signal[:, ppg_idx], record.p_signal[:, abp_idx], record.fs
    except Exception:
        return None


# ========== LOOP GLOBAL ==========
if __name__ == "__main__":
    if not os.path.exists(INPUT_CSV):
        print(f"Ficheiro {INPUT_CSV} não encontrado.")
        exit()

    df = pd.read_csv(INPUT_CSV)
    if PACIENTES_ESPECIFICOS:
        df = df[df['subject_folder'].astype(str).isin(PACIENTES_ESPECIFICOS)]
    pacientes = df['subject_folder'].astype(str).unique()[:MAX_PACIENTES]

    print(f"Visualização do Teste 1 (sem NeuroKit) | SQI: {'ATIVO' if USE_SQI_GATE else 'off'} (min={SQI_SKEW_MIN})")
    janelas_exportadas = []

    for s_id in pacientes:
        df_pac = df[df['subject_folder'].astype(str) == s_id]
        # 🆕 Primeiro passo: reúne TODAS as janelas candidatas do paciente (sem cortar por SQI
        # nem por MAX_JANELAS_POR_PACIENTE ainda) — o SQI só decide DEPOIS, por paciente,
        # exatamente como na secção 2.1. Sem isto, a retenção adaptativa não tem de onde escolher.
        candidatos_paciente = []

        for _, row in df_pac.iterrows():
            sinais = ler_record(row)
            if sinais is None: continue

            raw_ecg, raw_ppg, raw_abp, fs = sinais
            # Teste 1: tratar NaN e FILTRAR o sinal completo antes de janelar
            raw_ecg, raw_ppg, raw_abp = limpar_nan(raw_ecg), limpar_nan(raw_ppg), limpar_nan(raw_abp)
            raw_ecg = remover_picos_isolados(raw_ecg)  # 🆕 repara glitches de 1 amostra (dropout do ADC) antes do highpass
            ecg_filt = butter_highpass_filter(raw_ecg, 0.5, fs)
            if skew(ecg_filt) < 0:                # corrige ECG invertido (R para cima)
                ecg_filt = -ecg_filt
            ppg_filt = butter_lowpass_filter(raw_ppg, 8, fs)
            ppg_ac_sqi = butter_bandpass_filter(raw_ppg, 0.5, min(8, 0.45 * fs), fs)
            seg_name = str(row['segment'])

            samples_win  = int(WINDOW_SIZE_SEC * fs)
            samples_step = int(STRIDE_SEC * fs)

            if len(ecg_filt) < samples_win: continue
            n_windows = int((len(ecg_filt) - samples_win) / samples_step) + 1

            for w in range(n_windows):
                start, end = w * samples_step, w * samples_step + samples_win
                win_ecg, win_ppg, win_abp = ecg_filt[start:end], ppg_filt[start:end], raw_abp[start:end]

                # Gate original do Teste 1 (PPG plano / ABP sem pulsatilidade) — continua a ser
                # um corte rígido: indica janela genuinamente inutilizável, não "algo ruidosa".
                if np.std(win_ppg) < 0.005 or (np.max(win_abp) - np.min(win_abp)) < 15:
                    continue

                # 🆕 SQI de skewness — calculado SEMPRE, mas já não decide aqui (ver retenção
                # adaptativa por paciente logo a seguir ao loop de janelas, igual à secção 2.1).
                sqi_val = sqi_skewness(ppg_ac_sqi[start:end])

                # Gate ABP (targets fisiológicos) — igual ao Teste 2 viz
                sbp, dbp, sbp_std, pp = extrair_abp_targets(win_abp, fs)
                if sbp is None or not (pp > 25 and sbp_std < 12 and 65 < sbp < 195 and 35 < dbp < 115):
                    continue

                res = processar_janela_teste1(win_ecg, win_ppg, fs)
                if res is None or res[1] is None: continue

                beats, features, ecg_plot, ppg_clean = res
                ppg_plot = (ppg_clean - np.min(ppg_clean)).tolist()
                sbp_idxs, dbp_idxs = extrair_abp_markers(win_abp, fs)

                try: seg_order = int(seg_name.split('_')[-1])
                except: seg_order = 0

                candidatos_paciente.append({
                    'subject_id': s_id, 'segment_name': seg_name, 'segment_order': seg_order, 'window_idx': w,
                    'gender': row.get('gender', None), 'age': row.get('age', None), 'fs': float(fs),
                    'sqi_skew': round(float(sqi_val), 4) if np.isfinite(sqi_val) else None,
                    'ecg': [round(float(v), 5) for v in ecg_plot],
                    'ppg': [round(float(v), 5) for v in ppg_plot],
                    'abp': [round(float(v), 3) for v in win_abp],
                    'abp_markers': {'sbp_idxs': sbp_idxs, 'dbp_idxs': dbp_idxs},
                    'beats': beats,
                    'features': {**features, 'sbp_target': round(sbp, 2), 'dbp_target': round(dbp, 2)},
                })

        # --- 🆕 RETENÇÃO ADAPTATIVA POR PACIENTE (idêntica à secção 2.1) ---
        # Sem SQI: mantém tudo (Teste 1 puro). Com SQI: mantém sqi_skew>=SQI_SKEW_MIN, mas nunca
        # abaixo do piso (proporcional ao que o paciente tem + mínimo absoluto) — nunca zera
        # ninguém; se o corte normal não chegasse ao piso, relaxa para as melhores por skewness.
        n_total = len(candidatos_paciente)
        resgatado = False
        if USE_SQI_GATE and n_total > 0:
            piso = min(n_total, max(MIN_JANELAS_ABS, int(np.ceil(FRACAO_MINIMA_RETIDA * n_total))))
            ok = [c for c in candidatos_paciente if c['sqi_skew'] is not None and c['sqi_skew'] >= SQI_SKEW_MIN]
            if len(ok) >= piso:
                retidos = ok
            else:
                retidos = sorted(
                    candidatos_paciente,
                    key=lambda c: c['sqi_skew'] if c['sqi_skew'] is not None else -np.inf,
                    reverse=True,
                )[:piso]
                resgatado = True
        else:
            retidos = candidatos_paciente

        # Ordena cronologicamente (mais útil para inspeção visual) e corta ao limite de display
        retidos = sorted(retidos, key=lambda c: (c['segment_order'], c['window_idx']))[:MAX_JANELAS_POR_PACIENTE]

        for c in retidos:
            n_beats = len(c['beats'])
            skew_txt = f"{c['sqi_skew']:.2f}" if c['sqi_skew'] is not None else "n/d"
            print(f"  JANELA SALVA: {c['segment_name']} w{c['window_idx']:02d} | Batimentos: {n_beats} | skew={skew_txt}")
            janelas_exportadas.append(c)

        tag_resgate = " (resgatado pelo piso — corte normal deixaria menos janelas)" if resgatado else ""
        print(f"{s_id}: {n_total} candidatas → {len(retidos)} exportadas{tag_resgate}.")

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(to_native(janelas_exportadas), f)

    mb = os.path.getsize(OUTPUT_JSON) / (1024 * 1024)
    print(f"\nConcluído! {len(janelas_exportadas)} janelas. Ficheiro salvo em: {OUTPUT_JSON} ({mb:.1f} MB)")

## 2. Com SQI (Teste 1 aprimorado — SQI de skewness + retenção adaptativa por paciente)

### 2.1 Extração

In [ ]:
import os
import pandas as pd
import wfdb
import numpy as np
import re
from scipy.signal import find_peaks, butter, filtfilt
from scipy.integrate import simpson
from scipy.stats import skew
from joblib import Parallel, delayed
import multiprocessing

# --- CONFIGURAÇÃO ---
PATH_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"   # onde está o dataset bruto (leitura)
INPUT_CSVS = [f"{PATH_DATASET}/dataset_para_extração_features.csv"]
PATH_FEATURES_OUT = "/media/filipe/Dados/Features"               # onde gravar as features (escrita)
WINDOW_SIZE_SEC = 15
STRIDE_SEC = 5  # avanço de 5 s: as janelas de 15 s sobrepõem-se

# Nº de processos paralelos. O pico de RAM ≈ N_JOBS × (registo maior a ser processado). Alguns
# registos têm >1.5 GB; usar todos os cores (16) fazia colidir vários registos grandes e esgotar a
# RAM. 8 é um compromisso seguro (≈ metade dos cores). Pode subir-se com RAM folgada
# e descer-se se voltar a estourar. `None` = todos os cores (comportamento antigo, arriscado com este dataset).
N_JOBS = 16

# --- 🆕 QUALIDADE DE SINAL (SQI) — tunável: equilibra EXATIDÃO vs POPULAÇÃO ---
# SQI de skewness (Elgendi et al., 2016): melhor SQI de PPG no benchmark, invariante
# a escala/offset (ideal p/ unidades arbitrárias do MIMIC) e de custo ~zero.
#   USE_SQI_GATE = False  →  reproduz EXATAMENTE o Teste 1 original (baseline p/ comparação).
#   SQI_SKEW_MIN          →  sobe (0.3, 0.5) para + exatidão / - população; desce para o oposto.
USE_SQI_GATE = True
SQI_SKEW_MIN = 0.0
OUTPUT_SUFFIX = "com_sqi"   # ficheiros: features_{genero}_{OUTPUT_SUFFIX}.csv

# --- 🆕 REDE DE SEGURANÇA POR PACIENTE (retenção adaptativa) ---
# Achado empírico (2026-07-08, comparação kalman vs kalman_aprimorado, MAE holdout 6.52→7.18
# mmHg): dos 313 pacientes do baseline, os 73 que o corte SQI absoluto e global eliminava por
# completo tinham MENOS janelas de partida (mediana 146 vs 383, Mann-Whitney p=0.012) — não
# eram necessariamente piores, só tiveram menos "tentativas" para calhar numa janela com
# skewness positiva. Um corte cego e absoluto pune desproporcionalmente quem já tem pouco sinal.
# Correção: o SQI passa a ser calculado e exportado SEMPRE (coluna 'sqi_skew'), mas a decisão de
# manter/descartar janelas é feita DEPOIS, por paciente, com um piso mínimo — nunca zera ninguém.
FRACAO_MINIMA_RETIDA = 0.5   # nunca descarta mais de 50% das janelas válidas de um paciente por SQI
MIN_JANELAS_ABS = 20         # ...e nunca menos do que isto, se houver essa quantidade disponível

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def butter_highpass_filter(data, cutoff, fs, order=4):
    """Remove flutuações respiratórias da linha de base do ECG (Baseline Wander)"""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, data)

def butter_bandpass_filter(data, low, high, fs, order=4):
    """🆕 PPG-AC (0.5–8 Hz) usado SÓ para calcular o SQI de skewness.
    As features continuam a ser extraídas do ppg_filt (lowpass), logo os valores
    das features não mudam — o SQI apenas decide QUE janelas entram."""
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype='band', analog=False)
    return filtfilt(b, a, data)

def sqi_skewness(seg):
    """SQI de skewness (Elgendi 2016). Pulsos PPG bem formados (subida sistólica
    abrupta) são assimétricos à direita → skewness positiva. Ruído/artefacto/linha
    plana → skewness ~0 ou negativa. Invariante a escala e offset."""
    if seg is None or len(seg) < 10 or np.std(seg) < 1e-9:
        return -np.inf
    return float(skew(seg))

def remover_picos_isolados(sinal, limiar=0.15):
    """Repara glitches de UMA amostra (ex: dropout do ADC): marca a amostra i só se AMBOS os
    vizinhos imediatos divergem muito dela mas concordam ENTRE si — um QRS genuíno nunca tem essa
    assinatura (a subida ocupa várias amostras, vizinhos não concordam entre si). Substitui pela
    média dos vizinhos. Um filtro de mediana/MAD com janela maior corta os próprios picos R."""
    s = np.asarray(sinal, dtype=float)
    if len(s) < 3:
        return s
    d_ant = np.abs(s[1:-1] - s[:-2])
    d_prox = np.abs(s[1:-1] - s[2:])
    d_vizinhos = np.abs(s[:-2] - s[2:])
    glitch = (d_ant > limiar) & (d_prox > limiar) & (d_vizinhos < limiar * 0.5)
    limpo = s.copy()
    idx = np.where(glitch)[0] + 1
    limpo[idx] = (s[idx - 1] + s[idx + 1]) / 2
    return limpo

# Canal de ECG: EXCLUSIVAMENTE a derivação II (nome exato, não substring — 'V'/'I' soltos
# apanhavam CVP/ICP/IC2 por conterem essas letras). Nada de fallback para outras derivações: o
# setup real do projeto (AD8232, 3 elétrodos) mede Lead II, e treinar com a morfologia de QRS/T
# de outra derivação criaria uma discrepância treino↔uso real. Neste dataset todos os 9646
# registos têm II, por isso esta restrição não perde nenhum paciente.
def selecionar_ecg(sig_names):
    return sig_names.index('II') if 'II' in sig_names else None

def extrair_targets_abp_robusto(win_abp, fs):
    sbp_idxs, _ = find_peaks(win_abp, distance=int(fs*0.4), height=np.percentile(win_abp, 50))
    dbp_idxs, _ = find_peaks(-win_abp, distance=int(fs*0.4))

    if len(sbp_idxs) < 3 or len(dbp_idxs) < 3:
        return None, None, None, None

    sbp_vals = win_abp[sbp_idxs]
    sbp_mean = np.mean(sbp_vals)
    sbp_std = np.std(sbp_vals)
    dbp_mean = np.mean(win_abp[dbp_idxs])
    pulse_pressure = sbp_mean - dbp_mean

    return sbp_mean, dbp_mean, sbp_std, pulse_pressure


# ========== BEAT VAZIO ==========
def beat_vazio(r_now, q_idx, s_idx, t_idx, qn_idx, qrs_ok, qrs_duration, r_amplitude,
               qrs_amplitude, qrs_area, qt_interval, tq_interval, qtc_interval, sdi_interaction, hr):
    """Batimento sem pico PPG correspondente (ppg_pk é None). As métricas do lado ECG (Q/R/S/T,
    duração/amplitude do QRS, HR) já foram calculadas antes deste ponto e são independentes de
    haver pulso PPG a seguir — preservá-las evita marcar um QRS válido como 'excluído' só porque
    o PPG falhou (ex.: pulso fora da janela, ruído momentâneo no PPG)."""
    return {
        'r_idx': int(r_now), 'r_next_idx': None, 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
        'ppg_pk_idx': None, 'ppg_vly_idx': None, 'pat': None, 'hr': hr, 'pat_kept': False, 'qrs_ok': qrs_ok,
        'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude, 'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area,
        'qt_interval': qt_interval, 'tq_interval': tq_interval, 'qtc_interval': qtc_interval, 'sdi_interaction': sdi_interaction,
        'crest_time': None, 'decay_time': None, 'alpha': None, 'alpha_b': None, 'pulse_area': None, 'w50': None,
        'ppg_valley_intensity': None, 'ppg_ac_amplitude': None
    }


# ========== PIPELINE TESTE 1 (SEM NEUROKIT) — partilhado entre baseline/com_sqi/sem_sqi ==========
def processar_janela_teste1(win_ecg, win_ppg, fs):
    """Processa UMA janela com a lógica do Teste 1 (scipy, sem NeuroKit).
        win_ecg : ECG já filtrado (highpass 0.5 Hz, remove baseline wander)
        win_ppg : PPG já filtrado (lowpass 8 Hz)
    Devolve (beats, features, win_ecg, win_ppg). Os índices dos batimentos são relativos à janela.
    'r_next_idx' guarda o fim do pulso PPG (pe)."""
    win_ppg_pos = win_ppg - np.min(win_ppg) + 0.01   # positividade p/ alpha (não move os índices)

    # Deteção de picos R — abordagem HÍBRIDA (amplitude do ECG + envelope de Pan-Tompkins):
    #  • Candidatos por AMPLITUDE no próprio ECG (picos positivos ≥15% do máximo). Isto apanha
    #    de forma fiável todos os R genuínos, mesmo os de QRS menos afiado — um detetor puramente
    #    baseado no envelope (threshold global 0.3*max) perdia R verdadeiros quando o máximo do
    #    envelope estava inflado por uma onda S muito abrupta noutro ponto da janela.
    #  • O envelope (derivada²+integração ~80ms, realça a subida/descida rápida do QRS e suprime a
    #    onda T, mais lenta) serve para DESEMPATAR: quando dois candidatos ficam a <0.4s, fica o de
    #    maior envelope local (o R, sempre mais afiado) — mesmo que a onda T seja MAIS ALTA que o R
    #    (R atenuado). Assim resolve-se o problema clássico de marcar a T como R sem usar amplitude.
    deriv = np.diff(win_ecg, prepend=win_ecg[0]) * fs
    win_integ = max(1, int(0.08 * fs))
    envelope = np.convolve(deriv ** 2, np.ones(win_integ) / win_integ, mode='same')

    def _env_local(p):
        lo, hi = max(0, p - int(0.05 * fs)), min(len(envelope), p + int(0.05 * fs))
        return np.max(envelope[lo:hi])

    candidatos, _ = find_peaks(win_ecg, height=0.15 * np.max(win_ecg))
    r_peaks = []
    for p in candidatos:
        if r_peaks and (p - r_peaks[-1]) < int(fs * 0.4):
            env_p, env_last = _env_local(p), _env_local(r_peaks[-1])
            # Desempate: só troca por ENVELOPE quando a diferença é clara (>15%) — é o caso T-vs-R
            # (uma T mesmo alta tem sempre envelope MUITO menor, tipicamente >10x, porque é lenta).
            # Quando os envelopes ficam próximos, os dois candidatos pertencem à MESMA subida aguda
            # do QRS (ex.: um "ombro"/ruído no meio da subida antes do topo verdadeiro) — as janelas
            # de +-50ms usadas por _env_local sobrepõem-se o suficiente para "ver" a mesma transição
            # e empatar; aí o envelope não discrimina e a AMPLITUDE decide, porque é literalmente o
            # topo que procuramos. Sem isto, alguns batimentos ficavam presos num "ombro" da subida em
            # vez do pico real, sempre por empate exato do envelope entre dois pontos muito próximos.
            if env_p > env_last * 1.15 or (env_p > env_last / 1.15 and win_ecg[p] > win_ecg[r_peaks[-1]]):
                r_peaks[-1] = p
        else:
            r_peaks.append(int(p))
    r_peaks = np.array(r_peaks)

    # Onda T isolada detetada como R "solteiro" — quando o intervalo T-R (~QT+isoelétrico) cai
    # a >=0.4s da R verdadeira, ESCAPA ao desempate acima (que só compara candidatos <0.4s entre
    # si) e sobra como um "R" à parte. Rejeita só quando amplitude E envelope estão AMBOS muito
    # abaixo da mediana da própria janela — uma onda T mesmo alta nunca é afiada (T é lenta,
    # envelope baixo), mas um batimento genuíno de QRS largo/aberrante (ex.: PVC) mantém a
    # amplitude preservada mesmo com envelope mais baixo, por isso só uma condição não chega.
    # Uma onda T isolada (confirmada pelo padrão RR simétrico, que é independente) fica muito abaixo do
    # piso nas duas métricas; um batimento aberrante genuíno mantém a amplitude e sobrevive ao filtro.
    if len(r_peaks) >= 3:
        envs_all = np.array([_env_local(p) for p in r_peaks])
        amps_all = win_ecg[r_peaks]
        med_env, med_amp = np.median(envs_all), np.median(amps_all)
        if med_env > 0 and med_amp > 0:
            espurio = (envs_all < 0.15 * med_env) & (amps_all < 0.40 * med_amp)
            r_peaks = r_peaks[~espurio]

    if len(r_peaks) < 8:
        return None

    # Piso de amplitude mínima do R — o threshold relativo acima sempre encontra um "top 15%"
    # mesmo em ruído puro, porque é relativo ao próprio sinal. Um R genuíno (highpass 0.5Hz) tem
    # tipicamente amplitude bem acima de 0.1; abaixo de ~0.08 já não se distingue do ruído de
    # fundo, mesmo que a forma/distância pareçam plausíveis.
    amp_r_mediana = np.median(win_ecg[r_peaks])
    if amp_r_mediana < 0.08:
        return None

    # Segunda camada: a deteção híbrida encontra sempre "picos" mesmo em ruído puro, porque o
    # threshold é relativo ao máximo da janela — em ruído esses picos podem empurrar
    # amp_r_mediana para pouco acima do piso absoluto. A irregularidade do RR é o sinal mais
    # fiável de que não é um ritmo cardíaco real: batimentos genuínos têm RR consistente (mesmo
    # com arritmia leve, o CV é baixo); o ruído puro dá um CV muito maior. Só rejeita se
    # amplitude AINDA modesta (<0.15) E RR muito irregular — amplitude claramente forte não é
    # descartada só por variabilidade de RR mais alta.
    rr = np.diff(r_peaks) / fs
    cv_rr = np.std(rr) / np.mean(rr)
    if amp_r_mediana < 0.15 and cv_rr > 0.25:
        return None

    # Piso contra excursões anómalas na janela (artefacto de movimento/elétrodo — não é glitch de
    # 1 amostra, dura dezenas de ms, por isso `remover_picos_isolados` não o apanha): se algum
    # ponto da janela filtrada ultrapassa 4x a amplitude R típica DESTA janela, não é fisiológico.
    # Relativo à própria janela, não um valor fixo, porque a amplitude varia por registo/ganho.
    if np.max(np.abs(win_ecg)) > 4 * amp_r_mediana:
        return None

    prom_min = (np.max(win_ppg_pos) - np.min(win_ppg_pos)) * 0.15
    ppg_peaks_all, _ = find_peaks(win_ppg_pos, distance=int(fs * 0.30), prominence=prom_min)  # picos sistólicos globais
    rho_blood = 1060
    beats = []
    num_r = len(r_peaks)

    for i in range(num_r):
        r_now  = int(r_peaks[i])
        r_next = int(r_peaks[i + 1]) if i + 1 < num_r else None

        # --- Fiduciais ECG (Teste 1: argmin/argmax em janelas fixas) ---
        q_start = max(0, r_now - int(0.06 * fs))
        q_idx = int(q_start + np.argmin(win_ecg[q_start:r_now])) if q_start < r_now else r_now

        s_end_local = min(r_now + int(0.08 * fs), len(win_ecg))
        s_idx = int(r_now + np.argmin(win_ecg[r_now:s_end_local])) if r_now < s_end_local else r_now

        t_idx = None
        if r_next is not None:
            # margem antes do r_next (igual à do qn_idx) para o T nunca invadir a subida do QRS seguinte
            t_start = r_now + int(0.12 * fs)
            t_end = min(r_now + int(0.45 * fs), r_next - int(0.08 * fs))
            if t_start < t_end and t_start < len(win_ecg):
                t_idx = int(t_start + np.argmax(win_ecg[t_start:t_end]))

        qn_idx = None
        if r_next is not None:
            qn_start = max(r_now, r_next - int(0.06 * fs))
            if qn_start < r_next and qn_start < len(win_ecg):
                qn_idx = int(qn_start + np.argmin(win_ecg[qn_start:r_next]))

        # --- QRS ---
        qrs_duration = qrs_amplitude = qrs_area = r_amplitude = None
        qrs_ok = False
        if q_idx < s_idx < len(win_ecg):
            qrs_duration  = round((s_idx - q_idx) / fs, 4)
            qrs_ok        = 0.04 < qrs_duration < 0.16
            r_amplitude   = round(float(win_ecg[r_now]), 4)
            qrs_amplitude = round(float(win_ecg[r_now] - min(win_ecg[q_idx], win_ecg[s_idx])), 4)
            seg = win_ecg[q_idx:s_idx + 1] - win_ecg[q_idx]
            qrs_area = round(float(simpson(y=seg)), 4) if len(seg) > 1 else 0.0

        # --- QT / TQ / QTc / SDI ---
        qt_interval = tq_interval = qtc_interval = sdi_interaction = None
        if t_idx is not None and q_idx < t_idx < len(win_ecg):
            qt_interval = round((t_idx - q_idx) / fs, 4)
            if r_next is not None:
                rr_sec = (r_next - r_now) / fs
                qtc_interval = round(qt_interval / np.sqrt(rr_sec + 1e-6), 4)
        if t_idx is not None and qn_idx is not None and t_idx < qn_idx < len(win_ecg):
            tq_interval = round((qn_idx - t_idx) / fs, 4)
        if qt_interval is not None and tq_interval is not None:
            sdi_interaction = round(min(qt_interval / (tq_interval + 1e-6), 10.0), 4)

        # --- HR ---
        hr = round((60 * fs) / (r_next - r_now), 2) if r_next is not None else None

        # --- Pico sistólico PPG: 1º pico global depois do R (robusto a PAT curto/longo) ---
        lim = r_next if r_next is not None else min(r_now + int(0.9 * fs), len(win_ppg_pos))
        cand = ppg_peaks_all[(ppg_peaks_all > r_now) & (ppg_peaks_all <= lim)]
        ppg_pk = int(cand[0]) if len(cand) > 0 else None

        if ppg_pk is None:
            beats.append(beat_vazio(r_now, q_idx, s_idx, t_idx, qn_idx, qrs_ok, qrs_duration, r_amplitude,
                                     qrs_amplitude, qrs_area, qt_interval, tq_interval, qtc_interval, sdi_interaction, hr))
            continue

        pat = round((ppg_pk - r_now) / fs, 4)

        # --- Vale/pé PPG: mínimo ANTES do pico, limitado pelo pico PPG anterior (nunca pelo R) ---
        # Usar r_now como fronteira de início falha nos dois sentidos: com PAT curto, r_now cai
        # DEPOIS do pé verdadeiro (já na subida) e a busca nunca o alcança; com PAT grande
        # (latência do aparelho — ver janela de PAT adaptativa abaixo), r_now cai cedo demais e a
        # busca arrisca apanhar o vale de um ciclo anterior. O pé pertence à estrutura do PRÓPRIO PPG.
        picos_antes = ppg_peaks_all[ppg_peaks_all < ppg_pk]
        pe_ini = max(int(picos_antes[-1]) if len(picos_antes) > 0 else 0, ppg_pk - int(0.4 * fs), 0)
        ppg_vly = pe_ini + int(np.argmin(win_ppg_pos[pe_ini:ppg_pk])) if pe_ini < ppg_pk else max(0, ppg_pk - int(0.2 * fs))

        ppg_valley_intensity = float(win_ppg_pos[ppg_vly])
        ppg_ac_amplitude     = float(win_ppg_pos[ppg_pk] - win_ppg_pos[ppg_vly])
        omega     = (hr / 60.0) if hr is not None else 1.2
        alpha     = float(ppg_valley_intensity * np.sqrt((rho_blood * omega) / (ppg_ac_amplitude + 1e-6)))
        upslope   = win_ppg_pos[ppg_vly:ppg_pk + 1]
        deriv_ppg = np.diff(upslope) * fs
        max_slope = float(np.max(deriv_ppg)) if len(deriv_ppg) > 0 else 0.0
        b_coef    = 1.0 / (max_slope + 1e-6)
        alpha_b   = float(ppg_valley_intensity * np.sqrt((omega * rho_blood) / (b_coef + 1e-6)))

        crest_time = round((ppg_pk - ppg_vly) / fs, 4)

        # --- Fim do pulso PPG (Teste 1: primeiro mínimo local após o pico) ---
        # Limita a busca pelo PRÓXIMO PICO PPG (não pelo próximo R): usar r_next assumia PAT
        # curto, mas o PAT observado neste dataset pode ocupar quase todo o RR (latência do
        # aparelho de oximetria — ver janela de PAT adaptativa abaixo), o que deixava ppg_pk muito
        # perto de r_next e esganava esta busca (entalhe a colapsar em cima do próprio pico).
        prox_pico_ppg = ppg_peaks_all[ppg_peaks_all > ppg_pk]
        limite_pulso = int(prox_pico_ppg[0]) if len(prox_pico_ppg) > 0 else min(ppg_pk + int(0.9 * fs), len(win_ppg_pos))
        pe_start = ppg_pk + int(0.04 * fs)
        pe_end   = min(ppg_pk + int(0.50 * fs), len(win_ppg_pos), limite_pulso)
        if pe_start < pe_end:
            seg_pe = win_ppg_pos[pe_start:pe_end]
            mins, _ = find_peaks(-seg_pe, distance=int(fs * 0.05))
            pe = int(pe_start + mins[0]) if len(mins) > 0 else int(pe_start + np.argmin(seg_pe))
            decay_time = round((pe - ppg_pk) / fs, 4)
        else:
            pe = ppg_pk
            decay_time = None

        # --- Geometria do pulso ---
        p_low, p_high = max(0, ppg_pk - int(0.1 * fs)), min(len(win_ppg_pos), ppg_pk + int(0.3 * fs))
        pulso = win_ppg_pos[p_low:p_high]
        pulse_area = w50 = None
        if len(pulso) > 1 and (np.max(pulso) - np.min(pulso)) > 0.01:
            norm_p = (pulso - np.min(pulso)) / (np.max(pulso) - np.min(pulso))
            pulse_area = round(float(simpson(y=norm_p)), 4)
            idx50 = np.where(norm_p >= 0.5)[0]
            if len(idx50) > 1:
                w50 = round(float((idx50[-1] - idx50[0]) / fs), 4)

        # pat_kept definido depois do loop (janela de PAT adaptativa) — placeholder aqui
        pat_kept = False

        beats.append({
            'r_idx': r_now, 'r_next_idx': int(pe), 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
            'ppg_pk_idx': ppg_pk, 'ppg_vly_idx': ppg_vly, 'pat': pat, 'hr': hr,
            'pat_kept': pat_kept, 'qrs_ok': qrs_ok, 'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude,
            'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area, 'qt_interval': qt_interval, 'tq_interval': tq_interval,
            'qtc_interval': qtc_interval, 'sdi_interaction': sdi_interaction, 'crest_time': crest_time, 'decay_time': decay_time,
            'alpha': round(alpha, 4), 'alpha_b': round(alpha_b, 4), 'pulse_area': pulse_area, 'w50': w50,
            'ppg_valley_intensity': round(ppg_valley_intensity, 4), 'ppg_ac_amplitude': round(ppg_ac_amplitude, 4)
        })

    # --- Janela de PAT adaptativa por registo (substitui o intervalo fisiológico fixo 0.19-0.50s) ---
    # O PLETH tem latência de processamento própria do aparelho (média de vários batimentos p/
    # estabilizar o SpO2), consistente DENTRO do mesmo registo mas muito variável ENTRE registos —
    # confirmado por cross-check com o atraso R->ABP (medição direta, sem processamento de oxímetro),
    # que é apertado e estável, ao contrário do PAT do PPG cru. Em vez de um intervalo global fixo, aceitamos o agrupamento dominante e apertado
    # DESTE registo — seja qual for o valor absoluto — em bins de 40ms.
    pats_hr_ok = np.array([
        b['pat'] for b in beats
        if b['pat'] is not None and b['hr'] is not None and 45 < b['hr'] < 140
    ])
    if len(pats_hr_ok) >= 6:
        bins = np.arange(0, pats_hr_ok.max() + 0.04, 0.04)
        hist, edges = np.histogram(pats_hr_ok, bins=bins)
        i_max = int(np.argmax(hist))
        cl_min, cl_max = edges[i_max] - 0.04, edges[i_max + 1] + 0.04
        for b in beats:
            if b['pat'] is not None and b['hr'] is not None and 45 < b['hr'] < 140:
                b['pat_kept'] = bool(cl_min <= b['pat'] <= cl_max)

    # --- Filtragem IQR sobre os PATs aceites (Teste 1: remove outliers batimento a batimento) ---
    pats = np.array([b['pat'] for b in beats if b['pat'] is not None and b['pat_kept']])
    if len(pats) >= 4:
        q1, q3 = np.percentile(pats, [25, 75])
        iqr = (q3 - q1) if (q3 - q1) != 0 else 0.05
        lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        for b in beats:
            if b['pat'] is not None and b['pat_kept']:
                b['pat_kept'] = bool(lim_inf <= b['pat'] <= lim_sup)

    kept = [b for b in beats if b['pat_kept'] and b['pat'] is not None]
    kept_pats = np.array([b['pat'] for b in kept])
    # Teste 1 exige >= 6 batimentos limpos E baixa dispersão do PAT
    if len(kept) < 6 or np.std(kept_pats) >= 0.05:
        return beats, None, win_ecg, win_ppg

    def med(lst, key):
        vals = [b[key] for b in lst if b[key] is not None]
        return round(float(np.mean(vals)), 4) if vals else None

    features = {
        'pat_mean': med(kept, 'pat'), 'hr_mean': med(kept, 'hr'),
        'qrs_duration_mean': med([b for b in kept if b['qrs_ok']], 'qrs_duration'),
        'r_amplitude_mean':  med([b for b in kept if b['qrs_ok']], 'r_amplitude'),
        'qrs_amplitude_mean':med([b for b in kept if b['qrs_ok']], 'qrs_amplitude'),
        'qrs_area_mean':     med([b for b in kept if b['qrs_ok']], 'qrs_area'),
        'qt_interval_mean':  med([b for b in kept if b['qt_interval']  is not None], 'qt_interval'),
        'tq_interval_mean':  med([b for b in kept if b['tq_interval']  is not None], 'tq_interval'),
        'qtc_interval_mean': med([b for b in kept if b['qtc_interval'] is not None], 'qtc_interval'),
        'sdi_interaction_mean': med([b for b in kept if b['sdi_interaction'] is not None], 'sdi_interaction'),
        'crest_time_mean':   med(kept, 'crest_time'), 'decay_time_mean': med(kept, 'decay_time'),
        'alpha_mean':        med(kept, 'alpha'), 'alpha_b_mean': med(kept, 'alpha_b'),
        'pulse_area_mean':   med([b for b in kept if b['pulse_area'] is not None], 'pulse_area'),
        'w50_mean':          med([b for b in kept if b['pulse_area'] is not None], 'w50'),
        'ppg_valley_intensity_mean': med(kept, 'ppg_valley_intensity'),
        'ppg_ac_amplitude_mean':     med(kept, 'ppg_ac_amplitude'),
    }
    return beats, features, win_ecg, win_ppg


def processar_linha_individual(row):
    """
    Função atómica que processa apenas UMA linha do CSV original.
    Distribuída em paralelo por múltiplos cores.
    """
    s_id = str(row['subject_folder'])

    # --- DETEÇÃO AUTOMÁTICA DE DATASET ---
    if len(s_id) == 7:
        dataset_label = "MIMIC-III"
        canais_abp, canais_ppg = ['ABP', 'ART'], ['PLETH', 'PPG']
    elif len(s_id) == 9:
        dataset_label = "MIMIC-IV"
        canais_abp, canais_ppg = ['ABP'], ['PLETH']
    else:
        return []

    seg_name = str(row['segment'])
    try:
        seg_order = int(seg_name.split('_')[-1])
    except:
        seg_order = 0

    base_path = row['absolute_path'].replace('.dat', '').replace('.hea', '')
    if not os.path.exists(base_path + '.hea'):
        return []

    try:
        # MEMÓRIA: ler primeiro só o HEADER (barato) para descobrir os canais, e depois carregar
        # APENAS os 3 canais necessários (ECG/PPG/ABP) em vez do registo inteiro. Alguns registos
        # têm 6-8 canais e até 75h de duração (>1.5 GB carregados por inteiro em float64); carregar
        # só 3 canais + converter para float32 reduz o pico de RAM por worker para <1/4. Sem isto,
        # vários workers a apanhar registos grandes ao mesmo tempo esgotavam os 30 GB (OOM/crash).
        hdr = wfdb.rdheader(base_path)
        fs = hdr.fs
        sig_names = [s.upper() for s in hdr.sig_name]

        idx_ecg = selecionar_ecg(sig_names)
        idx_ppg = next((i for i, s in enumerate(sig_names) if any(x in s for x in canais_ppg)), None)
        idx_abp = next((i for i, s in enumerate(sig_names) if any(x in s for x in canais_abp)), None)
        if idx_ecg is None or idx_ppg is None or idx_abp is None:
            return []

        record = wfdb.rdrecord(base_path, channels=[idx_ecg, idx_ppg, idx_abp])
        ecg_raw = record.p_signal[:, 0].astype(np.float32)
        ppg_raw = record.p_signal[:, 1].astype(np.float32)
        abp_raw = record.p_signal[:, 2].astype(np.float32)
        del record  # liberta a matriz float64 devolvida pelo wfdb assim que temos as cópias float32

        if np.isnan(ecg_raw).any() or np.isnan(ppg_raw).any() or np.isnan(abp_raw).any():
            ecg_raw = pd.Series(ecg_raw).interpolate().ffill().bfill().values
            ppg_raw = pd.Series(ppg_raw).interpolate().ffill().bfill().values
            abp_raw = pd.Series(abp_raw).interpolate().ffill().bfill().values

        ecg_raw = remover_picos_isolados(ecg_raw)  # 🆕 repara glitches de 1 amostra (dropout do ADC) antes do highpass

        # Filtragem Biomédica dos Sinais
        ppg_filt = butter_lowpass_filter(ppg_raw, 8, fs)
        ecg_filt = butter_highpass_filter(ecg_raw, 0.5, fs)
        # Correção de inversão: se skew(ECG)<0 o sinal está invertido → repõe R para cima
        if skew(ecg_filt) < 0:
            ecg_filt = -ecg_filt
        # 🆕 PPG-AC só para o SQI de skewness (não afeta as features)
        ppg_ac_sqi = butter_bandpass_filter(ppg_raw, 0.5, min(8, 0.45 * fs), fs) if USE_SQI_GATE else None
    except Exception:
        return []

    resultados_da_linha = []

    # --- Janelas deslizantes com sobreposição ---
    samples_per_win = int(WINDOW_SIZE_SEC * fs)
    step_samples = int(STRIDE_SEC * fs)
    num_windows = int((len(ecg_filt) - samples_per_win) / step_samples) + 1

    for w in range(num_windows):
        start = w * step_samples
        end = start + samples_per_win

        win_ecg, win_ppg, win_abp = ecg_filt[start:end], ppg_filt[start:end], abp_raw[start:end]

        if np.std(win_ppg) < 0.005 or (np.max(win_abp) - np.min(win_abp)) < 15:
            continue

        # 🆕 SQI de skewness — calculado sempre (quando ativo) e exportado por janela. A
        # retenção/descarte NÃO é decidida aqui: é feita depois, por paciente, em
        # processar_datasets_hibridos() (retenção adaptativa com piso mínimo — ver CONFIG acima).
        sqi_val = sqi_skewness(ppg_ac_sqi[start:end]) if USE_SQI_GATE else np.nan

        # Gate ABP (targets fisiológicos) — computado cedo, antes do processamento pesado de batimentos
        sbp, dbp, sbp_std, pp = extrair_targets_abp_robusto(win_abp, fs)
        if sbp is None or not (pp > 25 and sbp_std < 12 and 65 < sbp < 195 and 35 < dbp < 115):
            continue

        res = processar_janela_teste1(win_ecg, win_ppg, fs)
        if res is None or res[1] is None:
            continue

        _, features, _, _ = res
        resultados_da_linha.append({
            'subject_id': s_id,
            'segment_name': seg_name,
            'segment_order': seg_order,
            'window_idx': w,
            'dataset': dataset_label,
            'gender': row['gender'],
            'age': row['age'],
            'pat_mean': features['pat_mean'],
            'hr_mean': features['hr_mean'],
            'qrs_duration': features['qrs_duration_mean'],
            'r_amplitude': features['r_amplitude_mean'],
            'qrs_amplitude': features['qrs_amplitude_mean'],
            'qrs_area': features['qrs_area_mean'],
            'w50': features['w50_mean'],
            'area_mean': features['pulse_area_mean'],
            'ppg_valley_intensity': features['ppg_valley_intensity_mean'],
            'ppg_ac_amplitude': features['ppg_ac_amplitude_mean'],
            'ppg_crest_time': features['crest_time_mean'],
            'ppg_decay_time': features['decay_time_mean'],
            'alpha_womersley': features['alpha_mean'],
            'alpha_womersley_b': features['alpha_b_mean'],
            'qt_interval': features['qt_interval_mean'],
            'tq_interval': features['tq_interval_mean'],
            'qtc_interval': features['qtc_interval_mean'],
            'sdi_interaction': features['sdi_interaction_mean'],
            'sbp_target': round(sbp, 2),
            'dbp_target': round(dbp, 2),
            'sqi_skew': round(float(sqi_val), 4) if np.isfinite(sqi_val) else np.nan
        })

    return resultados_da_linha

def processar_datasets_hibridos():
    for csv_file in INPUT_CSVS:
        if not os.path.exists(csv_file):
            print(f"Aviso: {csv_file} não encontrado.")
            continue

        df = pd.read_csv(csv_file)
        print(f"A inicializar processamento paralelo para {len(df)} registos em: {csv_file}")
        print(f" SQI: {'ATIVO' if USE_SQI_GATE else 'DESATIVADO (=baseline)'} | SQI_SKEW_MIN={SQI_SKEW_MIN}")

        num_cores = N_JOBS if N_JOBS is not None else multiprocessing.cpu_count()
        print(f"Motor de Processamento Paralelo ativado utilizando {num_cores} CORES (de {multiprocessing.cpu_count()} disponíveis).")

        raw_results = Parallel(n_jobs=num_cores, backend="multiprocessing", verbose=10, max_nbytes=None)(
            delayed(processar_linha_individual)(row) for _, row in df.iterrows()
        )

        features_list = []
        for res in raw_results:
            if res:
                features_list.extend(res)

        df_feat = pd.DataFrame(features_list)
        if not df_feat.empty:
            n_pac_antes = df_feat['subject_id'].nunique()
            n_janelas_antes = len(df_feat)

            # 🆕 RETENÇÃO ADAPTATIVA POR PACIENTE (ver CONFIG: FRACAO_MINIMA_RETIDA / MIN_JANELAS_ABS)
            # Substitui o corte SQI absoluto e cego. Cada paciente mantém as janelas com
            # sqi_skew >= SQI_SKEW_MIN; SE isso deixasse menos que o piso mínimo (proporcional
            # ao que o paciente tem + um mínimo absoluto), relaxa e fica com as `piso` melhores
            # por skewness em vez de zerar o paciente. Pacientes já "bons" não mudam de
            # comportamento — só os que o corte absoluto eliminaria por completo são resgatados.
            if USE_SQI_GATE:
                total_por_paciente = df_feat.groupby('subject_id')['sqi_skew'].transform('size')
                n_ok_por_paciente = df_feat.groupby('subject_id')['sqi_skew'].transform(
                    lambda s: (s >= SQI_SKEW_MIN).sum()
                )
                piso = np.minimum(
                    total_por_paciente,
                    np.maximum(MIN_JANELAS_ABS, np.ceil(FRACAO_MINIMA_RETIDA * total_por_paciente))
                ).astype(int)

                def _reter_por_paciente(g):
                    p = piso.loc[g.index].iloc[0]
                    ok = g['sqi_skew'] >= SQI_SKEW_MIN
                    if ok.sum() >= p:
                        return g[ok]
                    # Relaxa o limiar só o suficiente para atingir o piso deste paciente
                    return g.nlargest(p, 'sqi_skew')

                pacientes_resgatados = int((n_ok_por_paciente < piso).groupby(df_feat['subject_id']).any().sum())

                df_feat = (
                    df_feat.groupby('subject_id', group_keys=False)
                           .apply(_reter_por_paciente)
                           .reset_index(drop=True)
                )
                print(f"Retenção adaptativa: {n_janelas_antes}→{len(df_feat)} janelas | "
                      f"{n_pac_antes}→{df_feat['subject_id'].nunique()} pacientes | "
                      f"{pacientes_resgatados} pacientes resgatados pelo piso mínimo (não seriam zerados nem reduzidos abaixo dele).")

            os.makedirs(PATH_FEATURES_OUT, exist_ok=True)
            for g, label in [('M', 'homens'), ('F', 'mulheres')]:
                subset = df_feat[df_feat['gender'] == g]
                if not subset.empty:
                    # Guardar ficheiros finais segregados por género
                    subset.to_csv(f"{PATH_FEATURES_OUT}/features_{label}_{OUTPUT_SUFFIX}.csv", index=False)
            # 🆕 Relatório de retenção — monitoriza população vs baseline (356 pacientes)
            n_pac = df_feat['subject_id'].nunique()
            n_reg_ok = sum(1 for res in raw_results if res)
            print(f"SUCESSO PARALELO! Janelas totais processadas: {len(df_feat)}")
            print(f"Pacientes retidos: {n_pac} | Registos com ≥1 janela válida: {n_reg_ok}/{len(df)}")
            print(f"Média de janelas por paciente: {len(df_feat) / max(n_pac, 1):.1f}")
        else:
            print("Nenhuma janela válida gerada nesta execução.")

if __name__ == "__main__":
    processar_datasets_hibridos()


### 2.2 Visualização

Usa a mesma visualização do Teste 1 (ver secção **1.2**) — literalmente: `USE_SQI_GATE=True` em 1.2 aplica agora a mesma retenção adaptativa por paciente da secção 2.1 (nunca zera um paciente, ver piso mínimo), em vez do corte cego antigo. Antes desta correção (2026-07-09) não era verdade — a visualização só tinha o corte simples, dessincronizada da extração real.

## 3. Com NeuroKit (Teste final 2)

### 3.1 Extração

In [ ]:
import os
import pandas as pd
import wfdb
import numpy as np
from scipy.signal import find_peaks, butter, filtfilt
from scipy.integrate import simpson
from scipy.stats import skew
from joblib import Parallel, delayed
import multiprocessing

# --- CONFIGURAÇÃO ---
PATH_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"   # onde está o dataset bruto (leitura)
INPUT_CSVS = [f"{PATH_DATASET}/dataset_para_extração_features.csv"]
PATH_FEATURES_OUT = "/media/filipe/Dados/Features"               # 🆕 onde gravar as features (escrita)
WINDOW_SIZE_SEC = 15
STRIDE_SEC = 5

# Nº de processos paralelos. O pico de RAM ≈ N_JOBS × (registo maior + delineação NeuroKit por
# janela, que é pesada). Alguns registos têm >1.5 GB; 4 é um compromisso seguro (metade do que o
# baseline usa, porque aqui cada worker gasta mais). Pode subir-se com RAM folgada e descer-se se estourar.
N_JOBS = 4


# ========== FILTROS ==========
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return filtfilt(b, a, data)

def butter_highpass_filter(data, cutoff, fs, order=4):
    """Remove flutuações respiratórias da linha de base do ECG (Baseline Wander)"""
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='high', analog=False)
    return filtfilt(b, a, data)


# ========== DESPIKE + CANAL ECG ==========
def remover_picos_isolados(sinal, limiar=0.15):
    """Repara glitches de UMA amostra (ex: dropout do ADC): marca a amostra i só se AMBOS os
    vizinhos imediatos divergem muito dela mas concordam ENTRE si — um QRS genuíno nunca tem essa
    assinatura (a subida ocupa várias amostras, vizinhos não concordam entre si). Substitui pela
    média dos vizinhos. Aplicado antes de QUALQUER limpeza (inclui a própria nk.ecg_clean), já
    que estes glitches não são o tipo de ruído que os filtros de limpeza do NeuroKit visam."""
    s = np.asarray(sinal, dtype=float)
    if len(s) < 3:
        return s
    d_ant = np.abs(s[1:-1] - s[:-2])
    d_prox = np.abs(s[1:-1] - s[2:])
    d_vizinhos = np.abs(s[:-2] - s[2:])
    glitch = (d_ant > limiar) & (d_prox > limiar) & (d_vizinhos < limiar * 0.5)
    limpo = s.copy()
    idx = np.where(glitch)[0] + 1
    limpo[idx] = (s[idx - 1] + s[idx + 1]) / 2
    return limpo

# Canal de ECG: EXCLUSIVAMENTE a derivação II (nome exato, não substring — 'V'/'I' soltos
# apanhavam CVP/ICP/IC2 por conterem essas letras). Nada de fallback para outras derivações: o
# setup real do projeto (AD8232, 3 elétrodos) mede Lead II, e treinar com a morfologia de QRS/T
# de outra derivação criaria uma discrepância treino↔uso real. Neste dataset todos os 9646
# registos têm II, por isso esta restrição não perde nenhum paciente.
def selecionar_ecg(sig_names):
    return sig_names.index('II') if 'II' in sig_names else None


# ========== TARGETS ABP ==========
def extrair_targets_abp_robusto(win_abp, fs):
    sbp_idxs, _ = find_peaks(win_abp, distance=int(fs * 0.4), height=np.percentile(win_abp, 50))
    dbp_idxs, _ = find_peaks(-win_abp, distance=int(fs * 0.4))
    if len(sbp_idxs) < 3 or len(dbp_idxs) < 3:
        return 0, 0, 0, 0
    sbp_vals  = win_abp[sbp_idxs]
    sbp_mean  = np.mean(sbp_vals)
    sbp_std   = np.std(sbp_vals)
    dbp_mean  = np.mean(win_abp[dbp_idxs])
    pp        = sbp_mean - dbp_mean
    return sbp_mean, dbp_mean, sbp_std, pp


# ========== VALLEY CORRIGIDO: último mínimo local antes do pico sistólico ==========
def encontrar_valley(win_ppg, ppg_pk, nk_idx, ppg_peaks_win, nk_ppg_onsets_win, fs):
    """
    Procura o foot do pulso PPG como o último mínimo local imediatamente
    antes do pico sistólico, evitando capturar o vale do pico diastólico anterior.
    Todos os índices são relativos à janela.
    """
    ppg_vly = nk_ppg_onsets_win[nk_idx] if 0 <= nk_idx < len(nk_ppg_onsets_win) else None

    if ppg_vly is None or ppg_vly >= ppg_pk:
        # Busca desde o pico PPG anterior até ao pico atual
        if nk_idx > 0:
            search_start = ppg_peaks_win[nk_idx - 1]
        else:
            search_start = max(0, ppg_pk - int(0.8 * fs))
        search_start = max(0, search_start)

        if search_start < ppg_pk:
            seg = win_ppg[search_start:ppg_pk]
            # Último mínimo local = foot da subida sistólica
            local_mins, _ = find_peaks(-seg, distance=int(0.1 * fs))
            if len(local_mins) > 0:
                ppg_vly = int(search_start + local_mins[-1])
            else:
                restricted = max(search_start, ppg_pk - int(0.35 * fs))
                ppg_vly = int(restricted + np.argmin(win_ppg[restricted:ppg_pk]))
        else:
            ppg_vly = max(0, ppg_pk - int(0.15 * fs))

    return ppg_vly


# ========== PE CORRIGIDO: primeiro mínimo local após o pico sistólico ==========
def encontrar_pe(win_ppg, ppg_pk, nk_idx, ppg_peaks_win, nk_ppg_notches_win, r_next, fs):
    """
    Procura o fim do pulso PPG como o primeiro mínimo local imediatamente
    após o pico sistólico (notch dicrotico / início da diástole).
    Todos os índices são relativos à janela.
    """
    pe = nk_ppg_notches_win[nk_idx] if 0 <= nk_idx < len(nk_ppg_notches_win) else None

    if pe is None or pe <= ppg_pk:
        search_end = r_next if r_next is not None else min(ppg_pk + int(0.8 * fs), len(win_ppg) - 1)
        search_end = min(search_end, len(win_ppg) - 1)

        if ppg_pk < search_end:
            seg = win_ppg[ppg_pk:search_end]
            # Primeiro mínimo local = notch dicrotico / vale diastólico
            local_mins, _ = find_peaks(-seg, distance=int(0.1 * fs))
            if len(local_mins) > 0:
                pe = int(ppg_pk + local_mins[0])
            else:
                restricted = min(ppg_pk + int(0.35 * fs), search_end)
                pe = int(ppg_pk + np.argmin(win_ppg[ppg_pk:restricted]))
        else:
            pe = min(ppg_pk + int(0.25 * fs), len(win_ppg) - 1)

    return pe


# ========== FUNÇÃO PRINCIPAL (por linha do CSV) ==========
def processar_linha_individual(row):
    import neurokit2 as nk

    s_id = str(row['subject_folder'])

    if len(s_id) == 7:
        dataset_label = "MIMIC-III"
        canais_abp, canais_ppg = ['ABP', 'ART'], ['PLETH', 'PPG']
    elif len(s_id) == 9:
        dataset_label = "MIMIC-IV"
        canais_abp, canais_ppg = ['ABP'], ['PLETH']
    else:
        return []

    seg_name = str(row['segment'])
    try:
        seg_order = int(seg_name.split('_')[-1])
    except Exception:
        seg_order = 0

    base_path = row['absolute_path'].replace('.dat', '').replace('.hea', '')
    if not os.path.exists(base_path + '.hea'):
        return []

    # ── Leitura do registo ──────────────────────────────────────────────────────
    try:
        # MEMÓRIA: header primeiro (barato) para descobrir os canais, depois carregar APENAS os 3
        # canais necessários em float32 em vez do registo inteiro (6-8 canais, até 75h/>1.5 GB). Sem
        # isto, vários workers com registos grandes ao mesmo tempo esgotavam a RAM (OOM/crash).
        hdr = wfdb.rdheader(base_path)
        fs = hdr.fs
        sig_names = [s.upper() for s in hdr.sig_name]

        idx_ecg = selecionar_ecg(sig_names)
        idx_ppg = next((i for i, s in enumerate(sig_names) if any(x in s for x in canais_ppg)), None)
        idx_abp = next((i for i, s in enumerate(sig_names) if any(x in s for x in canais_abp)), None)
        if idx_ecg is None or idx_ppg is None or idx_abp is None:
            return []

        record = wfdb.rdrecord(base_path, channels=[idx_ecg, idx_ppg, idx_abp])
        ecg_raw = record.p_signal[:, 0].astype(np.float32)
        ppg_raw = record.p_signal[:, 1].astype(np.float32)
        abp_raw = record.p_signal[:, 2].astype(np.float32)
        del record
        ecg_raw = remover_picos_isolados(ecg_raw)  # 🆕 repara glitches de 1 amostra antes de qualquer limpeza
    except Exception:
        return []

    # ── NeuroKit2 no segmento COMPLETO — só limpeza + picos (leve) ─────────────
    # A delineação (pesada) é feita por janela para não explodir a RAM.
    try:
        # --- ECG: limpeza + deteção de picos R ---
        ecg_cleaned = nk.ecg_clean(ecg_raw, sampling_rate=fs, method="neurokit")
        if skew(ecg_cleaned) < 0:           # corrige inversão de sinal
            ecg_cleaned = -ecg_cleaned

        _, r_info = nk.ecg_peaks(ecg_cleaned, sampling_rate=fs, method="neurokit")
        r_peaks_global = r_info['ECG_R_Peaks']   # índices globais

        # --- PPG: limpeza + picos + qualidade ---
        ppg_signals, ppg_info = nk.ppg_process(ppg_raw, sampling_rate=fs, method="elgendi")
        ppg_cleaned         = ppg_signals["PPG_Clean"].values
        ppg_quality_global  = ppg_signals["PPG_Quality"].values
        nk_ppg_peaks_global = ppg_info["PPG_Peaks"].tolist()

    except Exception:
        return []

    resultados_da_linha = []
    samples_per_win = int(WINDOW_SIZE_SEC * fs)
    step_samples    = int(STRIDE_SEC * fs)
    num_windows     = int((len(ecg_cleaned) - samples_per_win) / step_samples) + 1

    # ── Loop de janelas deslizantes ─────────────────────────────────────────────
    for w in range(num_windows):
        start = w * step_samples
        end   = start + samples_per_win
        if end > len(ecg_cleaned):
            break

        # Rejeita janelas com PPG de baixa qualidade
        if np.mean(ppg_quality_global[start:end]) < 0.75:
            continue

        win_ecg = ecg_cleaned[start:end]
        win_ppg = ppg_cleaned[start:end]
        win_abp = abp_raw[start:end]
        win_ppg_pos = win_ppg - np.min(win_ppg) + 0.01   # garante positividade para alpha

        # Picos R na janela (índices globais e relativos)
        mask_r        = (r_peaks_global >= start) & (r_peaks_global < end)
        r_seg_indices = np.where(mask_r)[0]               # posições no array global
        r_peaks_win   = r_peaks_global[mask_r] - start    # relativos à janela

        if len(r_peaks_win) < 8:
            continue

        # Picos PPG na janela
        nk_ppg_arr        = np.array(nk_ppg_peaks_global)
        mask_ppg          = (nk_ppg_arr >= start) & (nk_ppg_arr < end)
        ppg_global_in_win = nk_ppg_arr[mask_ppg].tolist()
        ppg_peaks_win     = [p - start for p in ppg_global_in_win]

        # ── Delineação POR JANELA (leve: ~15s, ~15-20 batimentos) ─────────────
        try:
            _, waves_ecg = nk.ecg_delineate(win_ecg, r_peaks_win, sampling_rate=fs, method="dwt")

            def safe_ecg(k):
                raw = waves_ecg.get(k, [])
                return [int(x) if (x is not None and not np.isnan(x)) else None for x in raw]

            q_peaks_win = safe_ecg('ECG_Q_Peaks')
            s_peaks_win = safe_ecg('ECG_S_Peaks')
            t_peaks_win = safe_ecg('ECG_T_Peaks')
        except Exception:
            q_peaks_win, s_peaks_win, t_peaks_win = [], [], []

        try:
            _, waves_ppg = nk.ppg_delineate(win_ppg, ppg_peaks_win, sampling_rate=fs, method="derivative")

            def safe_ppg(k):
                raw = waves_ppg.get(k, [])
                return [int(x) if (x is not None and not np.isnan(x)) else None for x in raw]

            nk_ppg_onsets_win  = safe_ppg('PPG_Onsets')
            nk_ppg_notches_win = safe_ppg('PPG_Dicrotic_Notches')
        except Exception:
            nk_ppg_onsets_win, nk_ppg_notches_win = [], []

        # Lookup pico PPG (relativo à janela) → índice NeuroKit
        ppg_peak_to_nk_idx = {pk: i for i, pk in enumerate(ppg_peaks_win)}

        # Listas por batimento (sincronizadas entre si)
        b_pats, b_hrs           = [], []
        b_ppg_valleys, b_ppg_acs = [], []
        b_alphas, b_alphas_b    = [], []
        b_crest_times, b_decay_times = [], []
        b_w50, b_areas          = [], []
        # Listas independentes (condicionadas à validade do QRS / QT)
        b_qrs_durations, b_r_amplitudes, b_qrs_amplitudes, b_qrs_areas = [], [], [], []
        b_qts, b_tqs, b_qtcs, b_sdis = [], [], [], []

        # ── Loop de batimentos ─────────────────────────────────────────────────
        for i, r_seg_idx in enumerate(r_seg_indices):
            r      = int(r_peaks_win[i])
            r_next = int(r_peaks_win[i + 1]) if i + 1 < len(r_peaks_win) else None

            if r_next is None:
                continue

            hr = (60 * fs) / (r_next - r)
            if not (45 < hr < 140):
                continue

            # ── Q, S, T já relativos à janela (do delineamento por janela) ─────
            q_idx = q_peaks_win[i] if i < len(q_peaks_win) else None
            s_idx = s_peaks_win[i] if i < len(s_peaks_win) else None
            t_idx = t_peaks_win[i] if i < len(t_peaks_win) else None

            # Fallbacks Q e S
            if q_idx is None or not (0 <= q_idx < r):
                q_s   = max(0, r - int(0.05 * fs))
                q_idx = int(q_s + np.argmin(win_ecg[q_s:r])) if q_s < r else r
            if s_idx is None or not (r < s_idx < len(win_ecg)):
                s_e   = min(r + int(0.07 * fs), len(win_ecg))
                s_idx = int(r + np.argmin(win_ecg[r:s_e])) if r < s_e else r

            # ── QRS features ──────────────────────────────────────────────────
            if q_idx < s_idx < len(win_ecg):
                qrs_duration = (s_idx - q_idx) / fs
                if 0.04 < qrs_duration < 0.16:
                    r_amplitude   = float(win_ecg[r])
                    qrs_amplitude = float(win_ecg[r] - min(win_ecg[q_idx], win_ecg[s_idx]))
                    seg_qrs       = win_ecg[q_idx:s_idx + 1] - win_ecg[q_idx]
                    qrs_area      = float(simpson(y=seg_qrs)) if len(seg_qrs) > 1 else 0.0
                    b_qrs_durations.append(qrs_duration)
                    b_r_amplitudes.append(r_amplitude)
                    b_qrs_amplitudes.append(qrs_amplitude)
                    b_qrs_areas.append(qrs_area)

            # ── Intervalos QT, TQ, QTc ───────────────────────────────────────
            if t_idx is not None and q_idx is not None and q_idx < t_idx < len(win_ecg):
                qt_interval = (t_idx - q_idx) / fs
                rr_sec      = (r_next - r) / fs
                qtc         = qt_interval / np.sqrt(rr_sec + 1e-6)
                b_qts.append(qt_interval)
                b_qtcs.append(qtc)

                qn_idx = q_peaks_win[i + 1] if (i + 1) < len(q_peaks_win) else None
                if qn_idx is not None and t_idx < qn_idx < len(win_ecg):
                    tq_interval = (qn_idx - t_idx) / fs
                    b_tqs.append(tq_interval)
                    b_sdis.append(min(qt_interval / (tq_interval + 1e-6), 10.0))

            # ── PPG peak para este batimento ──────────────────────────────────
            ppg_candidates = [pk for pk in ppg_peaks_win if r < pk < r_next]
            if not ppg_candidates:
                continue

            ppg_pk = ppg_candidates[0]
            nk_idx = ppg_peak_to_nk_idx.get(ppg_pk, -1)

            # Valley e Pe com deteção de mínimos locais (índices relativos à janela)
            ppg_vly = encontrar_valley(win_ppg, ppg_pk, nk_idx,
                                       ppg_peaks_win, nk_ppg_onsets_win, fs)
            pe      = encontrar_pe(win_ppg, ppg_pk, nk_idx,
                                   ppg_peaks_win, nk_ppg_notches_win, r_next, fs)

            if not (ppg_vly < ppg_pk < pe):
                continue

            pat = (ppg_pk - r) / fs
            if not (0.19 < pat < 0.50):
                continue

            # ── Features sincronizadas (todas com o mesmo comprimento) ─────────
            b_pats.append(pat)
            b_hrs.append(hr)

            ppg_valley_intensity = float(win_ppg_pos[ppg_vly])
            ppg_ac_amplitude     = float(win_ppg_pos[ppg_pk] - win_ppg_pos[ppg_vly])
            omega                = hr / 60.0
            rho_blood            = 1060

            alpha = ppg_valley_intensity * np.sqrt((rho_blood * omega) / (ppg_ac_amplitude + 1e-6))

            upslope  = win_ppg_pos[ppg_vly:ppg_pk + 1]
            deriv    = np.diff(upslope) * fs
            max_slope = float(np.max(deriv)) if len(deriv) > 0 else 0.0
            alpha_b  = ppg_valley_intensity * np.sqrt((omega * rho_blood) / (1.0 / (max_slope + 1e-6) + 1e-6))

            b_ppg_valleys.append(ppg_valley_intensity)
            b_ppg_acs.append(ppg_ac_amplitude)
            b_alphas.append(alpha)
            b_alphas_b.append(alpha_b)

            b_crest_times.append((ppg_pk - ppg_vly) / fs)
            b_decay_times.append((pe - ppg_pk) / fs)

            # Geometria do pulso (área e w50)
            p_low  = max(0, ppg_pk - int(0.1 * fs))
            p_high = min(len(win_ppg_pos), ppg_pk + int(0.3 * fs))
            pulso  = win_ppg_pos[p_low:p_high]
            if len(pulso) > 1 and (np.max(pulso) - np.min(pulso)) > 0.01:
                norm_p = (pulso - np.min(pulso)) / (np.max(pulso) - np.min(pulso))
                b_areas.append(float(simpson(y=norm_p)))
                idx50 = np.where(norm_p >= 0.5)[0]
                if len(idx50) > 1:
                    b_w50.append((idx50[-1] - idx50[0]) / fs)

        # ── Filtragem IQR + agregação ──────────────────────────────────────────
        if len(b_pats) < 6:
            continue

        b_pats        = np.array(b_pats)
        b_hrs         = np.array(b_hrs)
        b_ppg_valleys = np.array(b_ppg_valleys)
        b_ppg_acs     = np.array(b_ppg_acs)
        b_alphas      = np.array(b_alphas)
        b_alphas_b    = np.array(b_alphas_b)
        b_crest_times = np.array(b_crest_times)
        b_decay_times = np.array(b_decay_times)

        q1, q3 = np.percentile(b_pats, [25, 75])
        iqr     = (q3 - q1) if (q3 - q1) != 0 else 0.05
        beats_limpos = (b_pats >= q1 - 1.5 * iqr) & (b_pats <= q3 + 1.5 * iqr)

        # Filtragem IQR independente para features QRS
        if len(b_r_amplitudes) >= 4:
            r_arr  = np.array(b_r_amplitudes)
            q1r, q3r = np.percentile(r_arr, [25, 75])
            iqr_r    = q3r - q1r
            mask_r_amp = (r_arr >= q1r - 2.0 * iqr_r) & (r_arr <= q3r + 2.0 * iqr_r)
            b_qrs_durations  = [v for v, ok in zip(b_qrs_durations,  mask_r_amp) if ok]
            b_r_amplitudes   = [v for v, ok in zip(b_r_amplitudes,   mask_r_amp) if ok]
            b_qrs_amplitudes = [v for v, ok in zip(b_qrs_amplitudes, mask_r_amp) if ok]
            b_qrs_areas      = [v for v, ok in zip(b_qrs_areas,      mask_r_amp) if ok]

        if np.sum(beats_limpos) < 6 or np.std(b_pats[beats_limpos]) >= 0.05:
            continue

        sbp, dbp, sbp_std, pp = extrair_targets_abp_robusto(win_abp, fs)
        if not (pp > 25 and sbp_std < 12 and 65 < sbp < 195 and 35 < dbp < 115):
            continue

        resultados_da_linha.append({
            'subject_id':          s_id,
            'segment_name':        seg_name,
            'segment_order':       seg_order,
            'window_idx':          w,
            'dataset':             dataset_label,
            'gender':              row['gender'],
            'age':                 row['age'],
            'pat_mean':            round(float(np.mean(b_pats[beats_limpos])), 4),
            'hr_mean':             round(float(np.mean(b_hrs[beats_limpos])), 2),
            'qrs_duration':        round(np.mean(b_qrs_durations),  4) if b_qrs_durations  else np.nan,
            'r_amplitude':         round(np.mean(b_r_amplitudes),   4) if b_r_amplitudes   else np.nan,
            'qrs_amplitude':       round(np.mean(b_qrs_amplitudes), 4) if b_qrs_amplitudes else np.nan,
            'qrs_area':            round(np.mean(b_qrs_areas),      4) if b_qrs_areas      else np.nan,
            'w50':                 round(np.mean(b_w50),            4) if b_w50            else np.nan,
            'area_mean':           round(np.mean(b_areas),          4) if b_areas          else np.nan,
            'ppg_valley_intensity':round(float(np.mean(b_ppg_valleys[beats_limpos])), 4),
            'ppg_ac_amplitude':    round(float(np.mean(b_ppg_acs[beats_limpos])),     4),
            'ppg_crest_time':      round(float(np.nanmean(b_crest_times[beats_limpos])), 4),
            'ppg_decay_time':      round(float(np.nanmean(b_decay_times[beats_limpos])), 4),
            'alpha_womersley':     round(float(np.mean(b_alphas[beats_limpos])),      4),
            'alpha_womersley_b':   round(float(np.mean(b_alphas_b[beats_limpos])),    4),
            'qt_interval':         round(np.mean(b_qts),  4) if b_qts  else np.nan,
            'tq_interval':         round(np.mean(b_tqs),  4) if b_tqs  else np.nan,
            'qtc_interval':        round(np.mean(b_qtcs), 4) if b_qtcs else np.nan,
            'sdi_interaction':     round(np.mean(b_sdis), 4) if b_sdis else np.nan,
            'sbp_target':          round(sbp, 2),
            'dbp_target':          round(dbp, 2),
        })

    return resultados_da_linha


# ========== ORQUESTRADOR PARALELO ==========
def processar_datasets_hibridos():
    for csv_file in INPUT_CSVS:
        if not os.path.exists(csv_file):
            print(f"Aviso: {csv_file} não encontrado.")
            continue

        df = pd.read_csv(csv_file)
        print(f"A inicializar processamento paralelo para {len(df)} registos em: {csv_file}")

        num_cores = N_JOBS if N_JOBS is not None else max(1, multiprocessing.cpu_count() // 2)
        print(f"Motor paralelo ativo: {num_cores} cores (de {multiprocessing.cpu_count()}; limitado para controlar RAM).")

        raw_results = Parallel(n_jobs=num_cores, backend="multiprocessing", verbose=10, max_nbytes=None)(
            delayed(processar_linha_individual)(row) for _, row in df.iterrows()
        )

        features_list = [item for res in raw_results if res for item in res]
        df_feat = pd.DataFrame(features_list)

        if not df_feat.empty:
            os.makedirs(PATH_FEATURES_OUT, exist_ok=True)
            for g, label in [('M', 'homens'), ('F', 'mulheres')]:
                subset = df_feat[df_feat['gender'] == g]
                if not subset.empty:
                    subset.to_csv(f"{PATH_FEATURES_OUT}/features_{label}_com_nk.csv", index=False)
            print(f"Concluído! Janelas totais processadas: {len(df_feat)}")
        else:
            print("Nenhuma janela válida gerada nesta execução.")


if __name__ == "__main__":
    processar_datasets_hibridos()

### 3.2 Visualização

In [ ]:
import os, json
import pandas as pd
import wfdb
import numpy as np
from scipy.signal import find_peaks
from scipy.integrate import simpson
from scipy.stats import skew  # Adicionado para detetar inversão de sinal
import neurokit2 as nk

# ========== CONFIGURAÇÃO ==========
PATH_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"
INPUT_CSV = f"{PATH_DATASET}/dataset_para_extração_features.csv"
OUTPUT_JSON = "/home/filipe/Desktop/Tese/algoritmo-ml/Visualização_e_Comparação/visualizacao_sinais_neurokit_total.json"

WINDOW_SIZE_SEC = 15
STRIDE_SEC = 10
MAX_PACIENTES = 6
MAX_JANELAS_POR_PACIENTE = 15
PACIENTES_ESPECIFICOS = None


# ========== DESPIKE + CANAL ECG ==========
def remover_picos_isolados(sinal, limiar=0.15):
    """Repara glitches de UMA amostra (ex: dropout do ADC): marca a amostra i só se AMBOS os
    vizinhos imediatos divergem muito dela mas concordam ENTRE si — um QRS genuíno nunca tem essa
    assinatura (a subida ocupa várias amostras, vizinhos não concordam entre si). Substitui pela
    média dos vizinhos. Aplicado antes de QUALQUER limpeza (inclui a própria nk.ecg_clean), já
    que estes glitches não são o tipo de ruído que os filtros de limpeza do NeuroKit visam."""
    s = np.asarray(sinal, dtype=float)
    if len(s) < 3:
        return s
    d_ant = np.abs(s[1:-1] - s[:-2])
    d_prox = np.abs(s[1:-1] - s[2:])
    d_vizinhos = np.abs(s[:-2] - s[2:])
    glitch = (d_ant > limiar) & (d_prox > limiar) & (d_vizinhos < limiar * 0.5)
    limpo = s.copy()
    idx = np.where(glitch)[0] + 1
    limpo[idx] = (s[idx - 1] + s[idx + 1]) / 2
    return limpo

# Canal de ECG: EXCLUSIVAMENTE a derivação II (nome exato, não substring — 'V'/'I' soltos
# apanhavam CVP/ICP/IC2 por conterem essas letras). Nada de fallback para outras derivações: o
# setup real do projeto (AD8232, 3 elétrodos) mede Lead II, e treinar/visualizar com a morfologia
# de QRS/T de outra derivação criaria uma discrepância treino↔uso real. Neste dataset todos os
# 9646 registos têm II, por isso esta restrição não perde nenhum paciente.
def selecionar_ecg(sig_names):
    return sig_names.index('II') if 'II' in sig_names else None


# ========== CONVERSÃO PARA TIPOS NATIVOS ==========
def to_native(obj):
    if isinstance(obj, (np.integer,)):   return int(obj)
    if isinstance(obj, (np.floating,)):  return float(obj)
    if isinstance(obj, np.ndarray):      return obj.tolist()
    if isinstance(obj, dict):            return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):   return [to_native(i) for i in obj]
    return obj


# ========== ABP MARKS ==========
def extrair_abp_markers(win_abp, fs):
    sbp_idxs, _ = find_peaks(win_abp,  distance=int(fs * 0.4), height=np.percentile(win_abp, 50))
    dbp_idxs, _ = find_peaks(-win_abp, distance=int(fs * 0.4))
    return [int(i) for i in sbp_idxs], [int(i) for i in dbp_idxs]

def extrair_abp_targets(win_abp, fs):
    sbp_idxs, dbp_idxs = extrair_abp_markers(win_abp, fs)
    if len(sbp_idxs) < 3 or len(dbp_idxs) < 3:
        return None, None, None, None
    sbp_vals = win_abp[sbp_idxs]
    dbp_vals = win_abp[dbp_idxs]
    sbp = float(np.mean(sbp_vals))
    dbp = float(np.mean(dbp_vals))
    sbp_std = float(np.std(sbp_vals))
    pp = sbp - dbp
    return sbp, dbp, sbp_std, pp


# ========== BEAT VAZIO ==========
def beat_vazio(r, q, s, t):
    return {
        'r_idx': int(r), 'r_next_idx': None, 'q_idx': q, 's_idx': s, 't_idx': t, 'qn_idx': None,
        'ppg_pk_idx': None, 'ppg_vly_idx': None, 'pat': None, 'hr': None, 'pat_kept': False, 'qrs_ok': False,
        'qrs_duration': None, 'r_amplitude': None, 'qrs_amplitude': None, 'qrs_area': None,
        'qt_interval': None, 'tq_interval': None, 'qtc_interval': None, 'crest_time': None, 'decay_time': None,
        'alpha': None, 'alpha_b': None, 'pulse_area': None, 'w50': None
    }


# ========== PIPELINE NEUROKIT2 AUTO-CORRETIVO ==========
def processar_janela_neurokit(win_ecg, win_ppg_raw, fs):
    
    METHOD_ECG_CLEAN     = "neurokit"     
    METHOD_ECG_PEAKS     = "neurokit"     
    METHOD_ECG_DELINEATE = "dwt"         

    # 1. Limpeza Inicial do ECG
    try:
        ecg_cleaned = nk.ecg_clean(win_ecg, sampling_rate=fs, method=METHOD_ECG_CLEAN)
        
        # CORREÇÃO DE INVERSÃO (A magia acontece aqui)
        # Se o skewness for negativo, significa que o sinal está "de pernas para o ar"
        if skew(ecg_cleaned) < 0:
            ecg_cleaned = -ecg_cleaned  # Inverte o sinal para ficar normalizado (R para cima)
            
        _, r_info = nk.ecg_peaks(ecg_cleaned, sampling_rate=fs, method=METHOD_ECG_PEAKS)
        r_peaks = r_info['ECG_R_Peaks'].tolist()
    except Exception as e:
        print(f"      Falha no processamento do ECG: {e}")
        return None

    if len(r_peaks) < 5:
        return None

    # 2. Delineação ECG
    try:
        _, waves_ecg = nk.ecg_delineate(ecg_cleaned, r_peaks, sampling_rate=fs, method=METHOD_ECG_DELINEATE)
        def safe_ecg(k): 
            raw = waves_ecg.get(k, [])
            return [int(x) if (x is not None and not np.isnan(x)) else None for x in raw]
        q_peaks, s_peaks, t_peaks = safe_ecg('ECG_Q_Peaks'), safe_ecg('ECG_S_Peaks'), safe_ecg('ECG_T_Peaks')
    except Exception:
        q_peaks, s_peaks, t_peaks = [], [], []

    # 3. Limpeza e Picos PPG (Elgendi)
    try:
        ppg_cleaned = nk.ppg_clean(win_ppg_raw, sampling_rate=fs, method="elgendi")
        _, ppg_info = nk.ppg_peaks(ppg_cleaned, sampling_rate=fs, method="elgendi")
        nk_ppg_peaks = ppg_info['PPG_Peaks'].tolist()
    except Exception:
        return None

    if len(nk_ppg_peaks) == 0:
        return None

    # 4. Delineação PPG (Derivative)
    try:
        _, waves_ppg = nk.ppg_delineate(ppg_cleaned, nk_ppg_peaks, sampling_rate=fs, method="derivative")
        def safe_ppg(k):
            raw = waves_ppg.get(k, [])
            return [int(x) if not np.isnan(x) else None for x in raw]
        nk_ppg_onsets  = safe_ppg('PPG_Onsets')
        nk_ppg_notches = safe_ppg('PPG_Dicrotic_Notches')
    except Exception:
        nk_ppg_onsets, nk_ppg_notches = [], []

    beats = []
    num_r = len(r_peaks)

    # 5. Loop de Batimentos
    for i, r in enumerate(r_peaks):
        r_next = r_peaks[i + 1] if i + 1 < num_r else None
        
        q_idx = q_peaks[i] if i < len(q_peaks) else None
        s_idx = s_peaks[i] if i < len(s_peaks) else None
        t_idx = t_peaks[i] if i < len(t_peaks) else None
        qn_idx = q_peaks[i + 1] if (i + 1) < len(q_peaks) else None

        if q_idx is None:
            q_s = max(0, r - int(0.05 * fs))
            q_idx = int(q_s + np.argmin(ecg_cleaned[q_s:r])) if q_s < r else r
        if s_idx is None:
            s_e = min(r + int(0.07 * fs), len(ecg_cleaned))
            s_idx = int(r + np.argmin(ecg_cleaned[r:s_e])) if r < s_e else r

        qrs_duration = qrs_amplitude = qrs_area = r_amplitude = None
        qrs_ok = False
        if q_idx < s_idx < len(ecg_cleaned):
            qrs_duration  = round((s_idx - q_idx) / fs, 4)
            qrs_ok        = 0.04 < qrs_duration < 0.16
            r_amplitude   = round(float(ecg_cleaned[r]), 4)
            qrs_amplitude = round(float(ecg_cleaned[r] - min(ecg_cleaned[q_idx], ecg_cleaned[s_idx])), 4)
            seg = ecg_cleaned[q_idx:s_idx + 1] - ecg_cleaned[q_idx]
            qrs_area = round(float(simpson(y=seg)), 4) if len(seg) > 1 else 0.0

        qt_interval = tq_interval = qtc_interval = None
        if t_idx is not None and q_idx < t_idx < len(ecg_cleaned):
            qt_interval = round((t_idx - q_idx) / fs, 4)
            if r_next is not None:
                rr_sec = (r_next - r) / fs
                qtc_interval = round(qt_interval / np.sqrt(rr_sec + 1e-6), 4)
        if t_idx is not None and qn_idx is not None and t_idx < qn_idx < len(ecg_cleaned):
            tq_interval = round((qn_idx - t_idx) / fs, 4)

        lim_sup_busca = r_next if r_next is not None else r + int(0.8 * fs)
        picos_candidatos = [idx for idx in nk_ppg_peaks if r < idx < lim_sup_busca]

        if not picos_candidatos:
            beats.append(beat_vazio(r, q_idx, s_idx, t_idx))
            continue

        ppg_pk = picos_candidatos[0]
        nk_idx = nk_ppg_peaks.index(ppg_pk)
        
        ppg_vly = nk_ppg_onsets[nk_idx] if nk_idx < len(nk_ppg_onsets) else None
        pe = nk_ppg_notches[nk_idx] if nk_idx < len(nk_ppg_notches) else None

        if ppg_vly is None or ppg_vly >= ppg_pk:
            if nk_idx > 0:
                search_start = nk_ppg_peaks[nk_idx - 1]
            else:
                search_start = max(0, ppg_pk - int(0.8 * fs))
            search_start = max(0, search_start)

            if search_start < ppg_pk:
                segment = ppg_cleaned[search_start:ppg_pk]
                # último mínimo local antes do pico = foot da subida sistólica
                local_mins, _ = find_peaks(-segment, distance=int(0.1 * fs))
                if len(local_mins) > 0:
                    ppg_vly = int(search_start + local_mins[-1])
                else:
                    # fallback: janela curta imediatamente antes do pico (só upstroke)
                    restricted = max(search_start, ppg_pk - int(0.35 * fs))
                    ppg_vly = int(restricted + np.argmin(ppg_cleaned[restricted:ppg_pk]))
            else:
                ppg_vly = max(0, ppg_pk - int(0.15 * fs))

        if pe is None or pe <= ppg_pk:
            search_end = r_next if r_next is not None else min(ppg_pk + int(0.8 * fs), len(ppg_cleaned) - 1)
            search_end = min(search_end, len(ppg_cleaned) - 1)

            if ppg_pk < search_end:
                segment = ppg_cleaned[ppg_pk:search_end]
                # primeiro mínimo local após o pico = vale diastólico/notch
                local_mins, _ = find_peaks(-segment, distance=int(0.1 * fs))
                if len(local_mins) > 0:
                    pe = int(ppg_pk + local_mins[0])
                else:
                    # fallback: janela curta imediatamente após o pico (só downstroke inicial)
                    restricted = min(ppg_pk + int(0.35 * fs), search_end)
                    pe = int(ppg_pk + np.argmin(ppg_cleaned[ppg_pk:restricted]))
            else:
                pe = min(ppg_pk + int(0.25 * fs), len(ppg_cleaned) - 1)

        hr = round((60 * fs) / (r_next - r), 2) if r_next is not None else 75.0

        ppg_valley_intensity = float(ppg_cleaned[ppg_vly])
        ppg_ac_amplitude = float(ppg_cleaned[ppg_pk] - ppg_cleaned[ppg_vly])
        omega = hr / 60.0
        rho_blood = 1060
        alpha = float(ppg_valley_intensity * np.sqrt((rho_blood * omega) / (ppg_ac_amplitude + 1e-6)))
        
        upslope = ppg_cleaned[ppg_vly:ppg_pk+1]
        deriv = np.diff(upslope) * fs
        max_slope = float(np.max(deriv)) if len(deriv) > 0 else 0.0
        b_coef = 1.0 / (max_slope + 1e-6)
        alpha_b = float(ppg_valley_intensity * np.sqrt((omega * rho_blood) / (b_coef + 1e-6)))

        crest_time = round((ppg_pk - ppg_vly) / fs, 4)
        decay_time = round((pe - ppg_pk) / fs, 4)

        p_low, p_high = max(0, ppg_pk - int(0.1*fs)), min(len(ppg_cleaned), ppg_pk + int(0.3*fs))
        pulso = ppg_cleaned[p_low:p_high]
        pulse_area = w50 = None
        if len(pulso) > 1 and (np.max(pulso) - np.min(pulso)) > 0.01:
            norm_p = (pulso - np.min(pulso)) / (np.max(pulso) - np.min(pulso))
            pulse_area = round(float(simpson(y=norm_p)), 4)
            idx50 = np.where(norm_p >= 0.5)[0]
            if len(idx50) > 1: w50 = round(float((idx50[-1] - idx50[0]) / fs), 4)

        beats.append({
            'r_idx': int(r), 'r_next_idx': int(pe), 'q_idx': q_idx, 's_idx': s_idx, 't_idx': t_idx, 'qn_idx': qn_idx,
            'ppg_pk_idx': int(ppg_pk), 'ppg_vly_idx': int(ppg_vly), 'pat': round((ppg_pk - r) / fs, 4), 'hr': hr if r_next is not None else None,
            'pat_kept': True, 'qrs_ok': qrs_ok, 'qrs_duration': qrs_duration, 'r_amplitude': r_amplitude, 
            'qrs_amplitude': qrs_amplitude, 'qrs_area': qrs_area, 'qt_interval': qt_interval, 'tq_interval': tq_interval, 'qtc_interval': qtc_interval,
            'crest_time': crest_time, 'decay_time': decay_time, 'alpha': round(alpha, 4), 'alpha_b': round(alpha_b, 4), 'pulse_area': pulse_area, 'w50': w50
        })

    pats = np.array([b['pat'] for b in beats if b['pat'] is not None])
    if len(pats) >= 4:
        q1, q3 = np.percentile(pats, [25, 75])
        iqr = (q3 - q1) if (q3 - q1) != 0 else 0.05
        lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        for b in beats:
            if b['pat'] is not None: b['pat_kept'] = bool(lim_inf <= b['pat'] <= lim_sup)

    kept = [b for b in beats if b['pat_kept'] and b['pat'] is not None]
    if len(kept) < 4:
        return beats, None, ecg_cleaned, ppg_cleaned

    def med(lst, key):
        vals = [b[key] for b in lst if b[key] is not None]
        return round(float(np.mean(vals)), 4) if vals else None

    features = {
        'pat_mean': med(kept, 'pat'), 'hr_mean': med(kept, 'hr'),
        'qrs_duration_mean': med([b for b in kept if b['qrs_ok']], 'qrs_duration'),
        'r_amplitude_mean':  med([b for b in kept if b['qrs_ok']], 'r_amplitude'),
        'qrs_amplitude_mean':med([b for b in kept if b['qrs_ok']], 'qrs_amplitude'),
        'qrs_area_mean':     med([b for b in kept if b['qrs_ok']], 'qrs_area'),
        'qt_interval_mean':  med([b for b in kept if b['qt_interval'] is not None], 'qt_interval'),
        'tq_interval_mean':  med([b for b in kept if b['qt_interval'] is not None], 'tq_interval'),
        'qtc_interval_mean': med([b for b in kept if b['qt_interval'] is not None], 'qtc_interval'),
        'crest_time_mean':   med(kept, 'crest_time'), 'decay_time_mean': med(kept, 'decay_time'),
        'alpha_mean':        med(kept, 'alpha'), 'alpha_b_mean': med(kept, 'alpha_b'),
        'pulse_area_mean':   med([b for b in kept if b['pulse_area'] is not None], 'pulse_area'),
        'w50_mean':          med([b for b in kept if b['pulse_area'] is not None], 'w50')
    }
    return beats, features, ecg_cleaned, ppg_cleaned


# ========== LEITURA DE CANAIS ==========
def ler_record(row):
    try:
        folder = str(row['subject_folder'])
        seg_name = str(row['segment'])
        path_record = f"{PATH_DATASET}/{folder}/{seg_name}"
        record = wfdb.rdrecord(path_record)
        sig_names = [s.upper() for s in record.sig_name]
        
        ppg_opts = ['PPG', 'PLETH']
        abp_opts = ['ABP', 'ART', 'PRESS']

        ecg_idx = selecionar_ecg(sig_names)
        ppg_idx = next((i for i, s in enumerate(sig_names) if any(x in s for x in ppg_opts)), None)
        abp_idx = next((i for i, s in enumerate(sig_names) if any(x in s for x in abp_opts)), None)

        if ecg_idx is None or ppg_idx is None or abp_idx is None:
            return None

        ecg_raw = remover_picos_isolados(record.p_signal[:, ecg_idx])  # 🆕 glitches de 1 amostra
        return ecg_raw, record.p_signal[:, ppg_idx], record.p_signal[:, abp_idx], record.fs
    except Exception:
        return None


# ========== LOOP GLOBAL ==========
if __name__ == "__main__":
    if not os.path.exists(INPUT_CSV):
        print(f"Ficheiro {INPUT_CSV} não encontrado.")
        exit()

    df = pd.read_csv(INPUT_CSV)
    if PACIENTES_ESPECIFICOS:
        df = df[df['subject_folder'].astype(str).isin(PACIENTES_ESPECIFICOS)]
    pacientes = df['subject_folder'].astype(str).unique()[:MAX_PACIENTES]

    print(f"Iniciando extração com Correção de Inversão por Skewness...")
    janelas_exportadas = []

    for s_id in pacientes:
        df_pac = df[df['subject_folder'].astype(str) == s_id]
        n_janelas = 0

        for _, row in df_pac.iterrows():
            if n_janelas >= MAX_JANELAS_POR_PACIENTE: break
            sinais = ler_record(row)
            if sinais is None: continue
                
            raw_ecg, raw_ppg, raw_abp, fs = sinais
            seg_name = str(row['segment'])
            
            samples_win  = int(WINDOW_SIZE_SEC * fs)
            samples_step = int(STRIDE_SEC * fs)
            
            if len(raw_ecg) < samples_win: continue
            n_windows = int((len(raw_ecg) - samples_win) / samples_step) + 1

            for w in range(n_windows):
                if n_janelas >= MAX_JANELAS_POR_PACIENTE: break

                start, end = w * samples_step, w * samples_step + samples_win
                win_ecg, win_ppg, win_abp = raw_ecg[start:end], raw_ppg[start:end], raw_abp[start:end]

                sbp, dbp, sbp_std, pp = extrair_abp_targets(win_abp, fs)
                if sbp is None or not (pp > 25 and sbp_std < 12 and 65 < sbp < 195 and 35 < dbp < 115):
                    continue

                res = processar_janela_neurokit(win_ecg, win_ppg, fs)
                if res is None or res[1] is None: continue
                    
                beats, features, ecg_clean, ppg_clean = res
                ppg_plot = (ppg_clean - np.min(ppg_clean)).tolist()
                sbp_idxs, dbp_idxs = extrair_abp_markers(win_abp, fs)

                print(f"  JANELA SALVA: {seg_name} w{w:02d} | Batimentos: {len(beats)}")

                try: seg_order = int(seg_name.split('_')[-1])
                except: seg_order = 0

                janelas_exportadas.append({
                    'subject_id': s_id, 'segment_name': seg_name, 'segment_order': seg_order, 'window_idx': w,
                    'gender': row.get('gender', None), 'age': row.get('age', None), 'fs': float(fs),
                    'ecg': [round(float(v), 5) for v in ecg_clean],
                    'ppg': [round(float(v), 5) for v in ppg_plot],
                    'abp': [round(float(v), 3) for v in win_abp],
                    'abp_markers': {'sbp_idxs': sbp_idxs, 'dbp_idxs': dbp_idxs},
                    'beats': beats,
                    'features': {**features, 'sbp_target': round(sbp, 2), 'dbp_target': round(dbp, 2)},
                })
                n_janelas += 1

        print(f"{s_id}: {n_janelas} janelas consolidadas.")

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(to_native(janelas_exportadas), f)

    mb = os.path.getsize(OUTPUT_JSON) / (1024 * 1024)
    print(f"\nConcluído! Sinais invertidos corrigidos matematicamente. Ficheiro salvo em: {OUTPUT_JSON} ({mb:.1f} MB)")